In [ ]:
import numpy as np
import pandas as pd
import pathlib
import textwrap
import time

import google.generativeai as genai
import ast
from IPython.display import display
from IPython.display import Markdown
import PIL.Image
import glob
from tqdm import tqdm
import random
import re

import vertexai
from vertexai.generative_models import GenerationConfig, Image, Part
import json

import csv
import os


np.random.seed(0)
# torch.manual_seed(0)
import random
random.seed(0)

In [ ]:
INPUT_TYPE = "numeric_only" #vision_only, numeric_and_vision

In [ ]:
from google import genai

client = genai.Client(api_key="REDACTED_GEMINI_API_KEY")

response = client.models.generate_content(
    model="gemini-1.5-flash",
    contents=[new_prompt],
)

print(response.text)

```csv
Professor Name;Proposal Name;Team Recommended
Brookshire, Robert G., Ph.D.;Advanced Technological Education;Brookshire, Robert G., Ph.D.;Dillon, Anthony L
Brookshire, Robert G., Ph.D.;Advanced Technological Education;Brookshire, Robert G., Ph.D.;Broughton, Earnie
Brookshire, Robert G., Ph.D.;Computer and Information Science and Engineering (CISE): Core Programs;Brookshire, Robert G., Ph.D.;Broughton, Earnie
Brookshire, Robert G., Ph.D.;Computer and Information Science and Engineering (CISE): Core Programs;Brookshire, Robert G., Ph.D.;Dillon, Anthony L
Brookshire, Robert G., Ph.D.;Division of Materials Research: Topical Materials Research Programs: Biomaterials (BMAT), Condensed Matter Physics (CMP), Metals and Metallic Nanostructures (MMN), Polymers (POL) (DMR-TMRP BMAT, CMP, MMN, POL) (nsf20589) | NSF - National Science Foundation;Brookshire, Robert G., Ph.D.;Broughton, Earnie
Brookshire, Robert G., Ph.D.;Division of Materials Research: Topical Materials Research Programs: Biom

In [ ]:
# Configure Gemini Client
GOOGLE_API_KEY = 'REDACTED_GEMINI_API_KEY'
client = genai.Client(api_key=GOOGLE_API_KEY)

# Extract all unique professors
def extract_all_professors(data):
    pattern = r"Professor Name: (.*?)\nSkill Set: ({.*?})"
    return re.findall(pattern, data, re.DOTALL)

# Get list of all professors
professor_entries = extract_all_professors(available_professors)
if len(professor_entries) < 46:
    raise ValueError("Expected at least 46 unique professors")

# Shuffle and split: 46 unique + 4 random (from full set)
random.shuffle(professor_entries)
unique_professors = professor_entries[:46]
remaining_randoms = [random.choice(professor_entries) for _ in range(4)]
# total_professors = unique_professors + remaining_randoms
total_professors = remaining_randoms

# Output directory
output_dir = "outputs_csv"
os.makedirs(output_dir, exist_ok=True)

# Generate prompts for 50 iterations
for i, (professor_name, skill_set) in enumerate(total_professors, start=1):
    safe_prof_name = professor_name.replace(",", "").replace(" ", "_")

    to_match_str = f"To Match:\n\nProfessor Name: {professor_name}\nSkill Set: {skill_set}"

    new_prompt = f"""{prompt2}

{to_match_str}

{available_professors}

{available_proposals}
"""

    try:
        response = client.models.generate_content(
            model="gemini-1.5-flash",
            contents=[new_prompt]
        )
        csv_content = response.text.strip()
        print(csv_content)

        # Parse and save the response (semicolon-separated)
        filepath = os.path.join(output_dir, f"{safe_prof_name}.csv")

        lines = csv_content.splitlines()
        if not lines:
            print(f"[{i}/50] ⚠️ Empty response for {safe_prof_name}")
            continue

        with open(filepath, "w", newline='', encoding="utf-8") as f:
            writer = csv.writer(f)
            for line in lines:
                parts = [part.strip() for part in line.split(';')]
                if len(parts) < 3:
                    continue
                prof_name = parts[0]
                proposal_name = parts[1]
                team_recommended = ';'.join(parts[2:])
                writer.writerow([prof_name, proposal_name, team_recommended])


        print(f"[{i}/50] ✅ Saved: {safe_prof_name}.csv")

    except Exception as e:
        print(f"[{i}/50] ❌ Error: {e}")

    time.sleep(4.1)  # Stay within rate limits


Professor Name;Proposal Name;Team Recommended
Alexeev, Oleg S.;Division of Materials Research: Topical Materials Research Programs: Biomaterials (BMAT), Condensed Matter Physics (CMP), Metals and Metallic Nanostructures (MMN), Polymers (POL) (DMR-TMRP BMAT, CMP, MMN, POL) (nsf20589) | NSF - National Science Foundation;Alexeev, Oleg S.; Chen, Fanglin (Frank)
Alexeev, Oleg S.;Division of Materials Research: Topical Materials Research Programs: Ceramics (CER), Electronic and Photonic Materials (EPM), Solid State and Materials Chemistry (SSMC);Alexeev, Oleg S.; Chao, Yuh J.
Alexeev, Oleg S.;Division of Chemistry: Disciplinary Research Programs;Alexeev, Oleg S.; Chen, Fanglin (Frank)
Alexeev, Oleg S.;Condensed Matter and Materials Theory (CMMT) (nsf20582) | NSF - National Science Foundation;Alexeev, Oleg S.; Chao, Yuh J.
Alexeev, Oleg S.;Division of Materials Research: Topical Materials Research Programs;Alexeev, Oleg S.; Chen, Fanglin (Frank)
Alexeev, Oleg S.;NSF/DOE Partnership in Basic P

In [ ]:
df = pd.DataFrame(columns=['Proposal Name', 'Professor Recommended'])
output = response.text.split('\n')[1:]

# Create a list to store the data
new_rows = []
for item in output:
  if item.strip():
    proposal, professor = item.split(';')
    # temp_list = professor_and_proposal.split(',')
    new_rows.append({'Proposal Name': proposal, 'Professor Recommended': professor})

# Use concat to append the new rows to the DataFrame
df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

In [ ]:
df.head()

,Proposal Name,Professor Recommended
0,Advanced Technological Education,"Broughton, Earnie"
1,Environmental Convergence Opportunities in Che...,"Bayoumi, Abdel-Moez E."
2,Collaborative Research in Computational Neuros...,"Boltin, Nicholas D."
3,National Artificial Intelligence (AI) Research...,"Ahmad, Iftikhar"
4,Computer and Information Science and Engineeri...,"Brookshire, Robert G., Ph.D."


In [ ]:
# save dataframe to csv file
df.to_csv('gemini_proposal_professor_recommendations_2.csv', index=False)

In [ ]:
df = pd.DataFrame(columns=['Professor Name', 'Proposal Name', 'Team Recommended'])
output = response.text.split('\n')[1:-1]

# Create a list to store the data
new_rows = []
for item in output:
    professor, proposal = item.split(';')[:2]
    # team = item.split(';')[2:] use join text
    team = ';'.join(item.split(';')[2:])
    new_rows.append({'Professor Name':professor, 'Proposal Name': proposal, 'Team Recommended': team})

# Use concat to append the new rows to the DataFrame
df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

In [ ]:
prompt2 = '''

You are an intelligent recommendation system designed for research collaborations. Your task is to recommend suitable teams and match them with the most appropriate proposals based on their expertise and specific evaluation metrics. Here’s the process:
Input:

1. Professor’s Name and Skill Set: I will provide a professor’s name along with their specific skills.
2. List of Available Professors and Their Skill Sets: You will receive a list of professors and their respective skills.
3. Available Proposals and Their Details: A list of research proposals with detailed descriptions.
Output:

1. Recommended Team: Based on the given professor’s skills and the available professors’ skills, recommend a team.
2. Proposal Matching: Match the recommended team with the most suitable research proposal.

Do not give any other details. Provide multiple outputs in a proper format of Professor Name; Proposal Name; Team Recommended. Use ';' to seperate the columns in the csv. Give multiple recommendations. Also, give multiple team recommendations for each professor and proposal combination. Please make sure that Team recommended should have multiple professors, not just one. Provide the recommendation output in CSV format.
'''

In [ ]:
available_professors = '''

Available Professors:

Professor Name: Agostinelli, Forest
Skill Set: {'search', 'artificial intelligence', 'reinforcement learning', 'bioinformatics', 'deep learning'}

Professor Name: Ahmad, Iftikhar
Skill Set: {'inc ph', 'computer simulation', 'aluminum content', 'semiconductor including', 'high aluminum', 'gallium oxide', 'previous position', 'ph texas', 'boron nitride', 'high power', 'bandgap semiconductor', 'including high', 'power electronic', 'ultra wide', 'novel high', 'universityresearch growth', 'electronic technology', 'boron nitride gallium oxide', 'position senior', 'nitride gallium', 'growth study', 'fabrication novel', 'content algan', 'sensor electronic', 'study ultra', 'fabrication novel high power electronic photonic device', 'inc ph texas tech universityresearch growth study ultra wide bandgap semiconductor including high aluminum content algan', 'computer simulation device', 'senior scientist', 'previous position senior scientist', 'sensor electronic technology', 'wide bandgap', 'simulation device', 'texas tech', 'photonic device', 'electronic photonic', 'tech universityresearch'}

Professor Name: Alexeev, Oleg S.
Skill Set: {'institute catalysis', 'exafs', 'catalysis considerable', 'cluster aggregate', 'used establish', 'gained technique', 'support interface', 'technique along', 'xrd', 'promoter structure', 'objective program', 'structure active site', 'various experimental', 'intermediate associated', 'site catalyst', 'spectroscopic technique', 'ph', 'adsorbed reactant', 'characterize active', 'structure metal cluster support', 'aggregate objective', 'active site', 'tpr', 'including tpd', 'catalyst adsorbed', 'combination various', 'particle size', 'property metal', 'specie actual', 'place sequence', 'adsorbed specie', 'russiacatalytic process surface take place sequence elementary reaction involving adsorbed reactant', 'catalyst surface', 'condition catalysis', 'chemisorption catalytic', 'boreskov institute catalysis', 'russiacatalytic process', 'establish dependence', 'structure active', 'focused fundamental', 'pursued application', 'xps', 'novosibirsk', 'boreskov institute', 'material spectroscopic', 'influence promoter structure catalytic property supported metal cluster aggregate objective program pursued application combination various experimental method characterization catalytic material spectroscopic technique', 'involving adsorbed', 'used characterize', 'influence promoter', 'fundamental understanding', 'structure metal', 'take place', 'product', 'actual condition', 'catalytic property', 'considerable information', 'property supported', 'metal cluster', 'used characterize active site catalyst adsorbed specie actual condition catalysis considerable information gained technique along chemisorption catalytic data used establish dependence catalytic property metal cluster particle size', 'novosibirsk state', 'program pursued', 'uv visible', 'supported metal', 'cluster support', 'hrtem', 'reaction involving', 'understanding structure', 'method characterization', 'surface goal', 'ftir', 'characterization catalytic', 'associated active', 'along chemisorption', 'surface take', 'sequence elementary', 'application combination', 'russiam', 'structure catalytic', 'process surface', 'cluster particle', 'elementary reaction', 'catalytic material', 'dependence catalytic', 'reaction intermediate', 'data used', 'structure metal support interface', 'metal support', 'reaction intermediate associated active site catalyst surface goal focused fundamental understanding structure metal support interface', 'goal focused', 'information gained', 'experimental method', 'catalytic data'}

Professor Name: Ali, Mohammod
Skill Set: {'usc july', 'director national', 'sc ph', 'nsf ipa', 'science foundation', 'dhaka received sc ph degree electrical engineering', 'mohammod ali', 'engineering usc', 'intergovernmental personnel', 'sc mohammod ali professor department electrical engineering usc july', 'canada', 'engineering bangladesh', 'dhaka received', 'assignment division', 'bangladesh engineering', 'foundation nsf', 'prof ali', 'ali received', 'department electrical', 'working program', 'ipa intergovernmental', 'engineering technology', 'sc mohammod', 'british columbia', 'program director', 'electrical electronic', 'respectively victoria', 'electrical engineering', 'received sc', 'ali professor', 'personnel act', 'division electrical', 'national science', 'electronic engineering', 'professor department', 'act assignment', 'main streetcolumbia', 'system directorate', 'swearingenroom main streetcolumbia', 'degree electrical', 'engineering prof', 'communication', 'cyber system', 'ph degree', 'ha working program director national science foundation nsf ipa intergovernmental personnel act assignment division electrical', 'swearingenroom main', 'ha working', 'sc electrical', 'directorate engineering', 'cyber system directorate engineering prof ali received sc electrical electronic engineering bangladesh engineering technology'}

Professor Name: Ammal, Salai C.
Skill Set: {'bharathidasan', 'ammal pdfph', 'cv dr', 'india', 'cv dr ammal pdfph', 'dr ammal'}

Professor Name: Bakos, Jason D.
Skill Set: {'high performance', 'computer architecture', 'performance computing', 'reconfigurable computing', 'heterogeneous computing', 'high performance computing', 'embedded system'}

Professor Name: Banerjee, Sourav
Skill Set: {'ultrasonics', 'acoustic', 'wave propagation', 'biomedical', 'metamaterials'}

Professor Name: Bayat, Mahmoud
Skill Set: {'nonlinear vibration', 'structural health', 'health monitoring', 'probabilistic analysis', 'earthquake eng', 'machine learning', 'structural health monitoring'}

Professor Name: Bayoumi, Abdel-Moez E.
Skill Set: {'various industry', 'predictive maintenance', 'engineering north', 'ha published', 'mechanical aerospace', 'mechanical material', 'ha received', 'sc bayoumi', 'experience currently', 'strong program', 'prediction mechanical', 'north state', 'bayoumi ha', 'engineering washington', 'industry experience', 'ha year', 'center predictive', 'life prediction', 'year teaching', 'actively involved', 'tribology', 'diagnosis prognosis life prediction mechanical system', 'relation professor', 'national science foundation various industry ha published journal conference paper', 'monitoring system', 'condition based maintenance', 'journal conference', 'sc bayoumi ha year teaching', 'state ha', 'director center', 'professor mechanical', 'project manager hewlett packard company', 'condition based', 'associate dean corporate relation professor mechanical biomedical engineering prior joining usc', 'biomedical', 'activity focused', 'manufacturing process', 'engineering activity', 'digital transformation', 'dean corporate', 'national science', 'wa professor mechanical aerospace engineering north state', 'developing strong', 'engineering prior', 'main streetroom', 'joining usc', 'aerospace center', 'based maintenance', 'prior joining', 'science foundation', 'funding department', 'corporate relation', 'received funding', 'foundation various', 'wa professor', 'washington state', 'transformation dr', 'director mcnair', 'mechanical biomedical', 'material engineering', 'currently director', 'nuclear aerospace', 'published journal', 'hewlett packard', 'industry ha', 'director center predictive maintenance', 'aerospace engineering', 'mechanical system', 'biomedical engineering', 'dr bayoumi', 'mechanical behavior', 'diagnosis prognosis', 'associate dean', 'health monitoring', 'health monitoring system', 'involved developing', 'conference paper', 'nuclear aerospace engineering activity focused mechanical behavior material', 'focused mechanical', 'prognosis life', 'industry experience currently director mcnair aerospace center', 'department energy', 'behavior material', 'department defense', 'ha actively', 'mcnair aerospace', 'program mechanical', 'manager hewlett', 'professor mechanical material engineering washington state ha actively involved developing strong program mechanical', 'digital transformation dr bayoumi ha received funding department defense', 'project manager', 'packard company'}

Professor Name: Benigni, Andrea
Skill Set: {'sc', 'swearingenroom main', 'swearingenroom main streetcolumbia', 'main streetcolumbia'}

Professor Name: Berge, Nicole D.
Skill Set: {'room berge', 'stream resource', 'process development', 'production specific', 'lead carbon', 'treatment process', 'solid waste', 'landfill thermochemical', 'berge focus', 'sustainable waste', 'personal care product endocrine disrupting compound bioreactor landfill thermochemical conversion municipal solid waste increasing energy yield waste stream resource recovery waste stream leachate treatment process development subsequent evaluation innovative groundwater remediation technology', 'technique lead', 'yield waste', 'innovative groundwater', 'promote sustainable', 'value added', 'room berge focus improving understanding physical', 'energy generation', 'biological process', 'leachate treatment', 'product endocrine', 'pharmaceutical', 'value added product production specific area exploration fate disposed nanomaterials', 'waste treatment', 'development subsequent', 'energy yield', 'main street', 'waste increasing', 'understanding physical', 'treatment technique', 'added product', 'recovery waste', 'carbon sequestration', 'bioreactor landfill', 'evaluation innovative', 'focus improving', 'exploration fate', 'disrupting compound', 'endocrine disrupting', 'conversion municipal', 'product production', 'area exploration', 'manipulated promote', 'stream leachate', 'chemical', 'biological process manipulated promote sustainable waste treatment technique lead carbon sequestration', 'care product', 'municipal solid', 'specific area', 'remediation technology', 'personal care', 'improving understanding', 'fate disposed', 'groundwater remediation', 'waste stream', 'increasing energy', 'thermochemical conversion', 'subsequent evaluation', 'compound bioreactor', 'resource recovery', 'process manipulated', 'disposed nanomaterials'}

Professor Name: Besmann, Theodore  M.
Skill Set: {'nuclear fuel', 'fuel waste', 'waste material', 'nuclear fuel waste material'}

Professor Name: Bischoff, Jeff
Skill Set: {'mechanic soft', 'soft tissue', 'analysis structural constitutive evolution artificial tissue scaffolding', 'bischoff biomedical', 'passive mechanic', 'modeling growth', 'artificial tissue', 'growth tissue', 'constitutive evolution', 'dr bischoff', 'tissue scaffolding', 'hypertrophic scar', 'theoretical framework modeling growth tissue', 'framework modeling', 'passive mechanic soft tissue', 'testing modeling', 'evolution artificial', 'theoretical framework', 'modeling hypertrophic', 'dr bischoff biomedical engineering', 'biomedical engineering', 'constitutive testing', 'constitutive testing modeling hypertrophic scar', 'structural constitutive', 'analysis structural'}

Professor Name: Blanchette, James  Otto
Skill Set: {'specific project', 'developing responsive', 'islet cell', 'impact induction', 'pre angiogenic', 'cell cell', 'cell enhance', 'vascularization skiles', 'room main streetcolumbia', 'skiles l', 'orally extending', 'geometry culture', 'therapeutic tissue', 'chemotherapeutics administered', 'dimensional cell', 'instructive material', 'material tracking', 'brougham cook blanchette asc spheroid geometry culture oxygenation differentially impact induction pre angiogenic behavior endothelial cell cell transplantation', 'fall area', 'transplantation mesenchymal', 'stem cell', 'genetic engineering', 'extending proper', 'function pancreatic', 'engineering design', 'blanchette asc', 'asc spheroid', 'angiogenic behavior', 'spectrum chemotherapeutics', 'delivery therapeutic tissue engineering specific project developing responsive delivery system expand spectrum chemotherapeutics administered orally extending proper function pancreatic islet cell following transplantation genetic engineering design instructive hydrogel material tracking hypoxic signaling three dimensional cell aggregate transplantation mesenchymal stem cell enhance vascularization skiles l', 'following transplantation', 'spheroid geometry', 'aggregate transplantation', 'induction pre', 'sc fall', 'cook blanchette', 'area design', 'hanna', 'room main', 'system expand', 'proper function', 'delivery therapeutic', 'instructive hydrogel', 'transplantation genetic', 'design instructive', 'cell transplantation', 'cell instructive', 'delivery system', 'behavior endothelial', 'engineering specific', 'main streetcolumbia', 'sc fall area design cell instructive material', 'hypoxic signaling', 'mesenchymal stem', 'differentially impact', 'endothelial cell', 'project developing', 'tissue engineering', 'three dimensional', 'responsive delivery', 'signaling three', 'cell following', 'enhance vascularization', 'cell aggregate', 'tracking hypoxic', 'brougham cook', 'culture oxygenation', 'oxygenation differentially', 'administered orally', 'design cell', 'tipton', 'rucker', 'hydrogel material', 'expand spectrum', 'pancreatic islet'}

Professor Name: Boltin, Nicholas D.
Skill Set: {'predictive modeling', 'data mining', 'glean information', 'mass casualty', 'valdes', 'incident edit', 'design evaluate translational informatic tool created deploy sophisticated algorithm using latest software development human computer interaction boltin', 'application dimensional', 'network improve', 'focus developing', 'statistic', 'dimension reduction', 'article press', 'study jmir', 'focus categorized', 'initial study', 'valafar mobile decision support tool emergency department mass casualty incident edit initial study jmir mhealth uhealth', 'well predictive modeling two', 'jmir mhealth', 'including data', 'latest software', 'software development', 'supervised unsupervised', 'sophisticated algorithm', 'employ technique', 'department triage', 'mobile decision', 'science focus', 'created deploy', 'support tool', 'triage chemical', 'algorithm using', 'information data', 'unsupervised machine', 'system healthcare', 'incident article', 'using latest', 'culley jm', 'development human', 'casualty incident', 'artificial neural', 'science computer', 'broad area', 'culley', 'interdisciplinary field', 'improve emergency', 'science interdisciplinary', 'department mass', 'dr boltin', 'tool emergency', 'valafar application dimensional reduction artificial neural network improve emergency department triage chemical mass casualty incident article press boltin', 'mhealth uhealth', 'technique including', 'developing decision', 'discipline mathematics', 'reduction artificial', 'interaction boltin', 'valafar application', 'chemical mass', 'dr boltin focus developing decision support system healthcare industry data science interdisciplinary field employ technique many discipline mathematics', 'translational informatic', 'two broad', 'industry data', 'informatic tool', 'deploy sophisticated', 'healthcare industry', 'well predictive', 'decision support', 'information science computer science focus categorized two broad area one', 'design evaluate', 'technique many', 'categorized two', 'many discipline', 'area one', 'tool created', 'support system', 'press boltin', 'modeling two', 'machine learning', 'statistical analysis', 'emergency department', 'supervised unsupervised machine learning', 'computer interaction', 'field employ', 'glean information data utilizing analytical technique including data mining', 'information science', 'evaluate translational', 'neural network', 'valafar mobile', 'boltin focus', 'utilizing analytical', 'data science', 'dimensional reduction', 'edit initial', 'analytical technique', 'data utilizing', 'human computer', 'computer science'}

Professor Name: Booth, Kristen
Skill Set: {'medium frequency transformer optimization', 'frequency transformer', 'power electronics', 'power grid', 'medium frequency', 'power grid resiliency', 'transformer optimization', 'electronics reliability', 'grid resiliency', 'power electronics reliability'}

Professor Name: Brookshire, Robert G., Ph.D.
Skill Set: {'european journal operational', 'co author', 'taught new', 'performance journal itec record managementitec mainframe systemsitec linux programming administration', 'article appeared', 'north texas', 'organizational system', 'columbia', 'sc robert brookshire professor integrated information technology department college engineering computing', 'legislative study', 'european journal', 'journal past', 'system association', 'editor information', 'managementitec mainframe', 'appeared journal', 'journal past president organizational system association editor information technology', 'innovation assembly street', 'social science computer review', 'director master', 'article appeared journal computer information system', 'suite', 'master health', 'learning', 'sage publication', 'study quarterly', 'sc director master health information technology program ha taught new york', 'integrated information', 'james madison', 'journal computer', 'health information', 'author using', 'innovation assembly', 'president organizational', 'department college', 'virginia', 'texas state', 'college engineering', 'past president', 'record managementitec', 'programming administration', 'james madison co author using microcomputer sage publication', 'assembly street', 'information technology', 'professor integrated', 'technology program', 'linux programming', 'technology department', 'legislative study quarterly', 'information system', 'mainframe systemsitec', 'sc director', 'ha taught', 'brookshire professor', 'journal itec', 'association editor', 'computer review', 'social science', 'new york', 'science computer', 'performance journal', 'sc robert', 'engineering computing', 'computer information', 'north texas state', 'microcomputer sage', 'program ha', 'robert brookshire', 'madison co', 'byte', 'using microcomputer', 'journal operational', 'systemsitec linux', 'itec record'}

Professor Name: Broughton, Earnie
Skill Set: {'lead software', 'practice responsibility', 'gathering stakeholder', 'earnie lead software developer virtual test bed vtb smart ship system design project addition writing code', 'development team', 'smart ship', 'esrdc consortium', 'virtual test', 'daily activity', 'vtb smart', 'test bed', 'student programming', 'consortium education', 'project addition', 'design project', 'activity development', 'including mentoring', 'development best', 'software developer', 'industry requirement', 'writing code', 'responsibility gathering', 'professional development', 'plan integrating', 'developer virtual', 'mentoring student', 'team project', 'private industry', 'best practice', 'private industry requirement developing plan integrating requirement consortium education', 'system design', 'bed vtb', 'integrating requirement', 'requirement developing', 'programming professional', 'supervises daily', 'ship system', 'developing plan', 'including mentoring student programming professional development best practice responsibility gathering stakeholder navy', 'earnie lead', 'supervises daily activity development team project', 'addition writing', 'requirement consortium', 'stakeholder navy'}

Professor Name: Buell, Duncan A.
Skill Set: {'work ha', 'arithmetic often', 'pedagogy professor', 'oriented toward', 'ward one', 'one often need substantial parallelism', 'iphone urban', 'broad class', 'algorithm architecture', 'improvement writing', 'digital humanity', 'often multiprecision', 'evicting resident', 'project building', 'mathematics text', 'professor buell', 'co supervising', 'supervising work', 'traditional computer architecture oriented toward floating point computation normally perform significantly reduced efficiency one class problem computational problem number theory fast integer arithmetic', 'voting system', 'included algorithm', 'one neighborhood', 'machine used', 'theory fast', 'number theory', 'integer arithmetic', 'vote total', 'support vote', 'string processing', 'recent work', 'buell past', 'architecture performing', 'computer architecture', 'significantly reduced', 'ha part', 'bit arithmetic', 'directed splash', 'first year', 'floating point', 'done string', 'traditional computer', 'process lead', 'substantial parallelism', 'electronic voting', 'building custom', 'devised bit', 'machine devised', 'cc directed', 'team auditing', 'data discovering', 'whose application', 'efficiency one', 'essay linguistic', 'discrete mathematics text string processing', 'often multiprecision arithmetic', 'compute element', 'uncounted vote', 'discrete mathematics', 'splash project', 'critical interactive', 'bit bit', 'necessary another broad class problem custom computing machine devised bit oriented computation done string processing', 'vote absence', 'past included', 'text string', 'critical interactive iphone urban renewal columbia sc role evicting resident ward one neighborhood adjacent', 'reduced efficiency', 'oriented computation', 'performing computation', 'system digital', 'election data', 'ha electronic', 'point computation', 'auditing election', 'used xilinx', 'problem process', 'one often', 'part team', 'often sufficient', 'work ward', 'multiprecision arithmetic', 'one class', 'element whose', 'often need', 'sc role', 'professor buell recent work ha electronic voting system digital humanity ha part team auditing election data discovering problem process lead uncounted vote absence data support vote total certified state digital humanity', 'linguistic characteristic', 'columbia sc', 'toward floating', 'interactive iphone', 'year english', 'fast integer', 'custom computing', 'class problem', 'computation normally', 'data support', 'application programmed', 'one app', 'computational problem', 'accuracy ida', 'another broad', 'ida cc', 'absence data', 'need substantial', 'renewal columbia', 'programmed vhdl', 'discovering problem', 'resident ward', 'architecture oriented', 'characteristic improvement', 'total certified', 'urban renewal', 'problem custom', 'processing computation', 'engaged analysis first year english essay linguistic characteristic improvement writing pedagogy professor buell past included algorithm architecture performing computation', 'role evicting', 'fpgas compute', 'xilinx fpgas', 'buell recent', 'sufficient accuracy', 'neighborhood adjacent', 'co supervising work ward one app', 'computing machine', 'problem computational', 'image processing', 'writing pedagogy', 'certified state', 'computation done', 'perform significantly', 'problem number', 'lead uncounted', 'bit oriented', 'image processing computation', 'necessary another', 'bit bit arithmetic often sufficient accuracy ida cc directed splash project building custom computing machine used xilinx fpgas compute element whose application programmed vhdl', 'english essay', 'engaged analysis', 'analysis first', 'normally perform', 'humanity ha', 'state digital'}

Professor Name: Caicedo, Juan
Skill Set: {'george brown', 'dynamic intelligent', 'mechanic american', 'structural health', 'earthquake engineering structural control dr caicedo member american society civil engineer', 'structural dynamic', 'dr caicedo', 'health monitoring', 'caicedo member', 'area specialization', 'intelligent infrastructure', 'structural health monitoring', 'civil engineer', 'earthquake engineering', 'project publication', 'experimental area', 'area structural', 'brown jr', 'visit structural', 'structural control', 'structural engineering', 'engineering structural', 'caicedo area', 'control dr', 'infrastructure lab', 'model updating', 'society experimental', 'engineering educator', 'george brown jr network earthquake engineering simulation', 'list project', 'engineering institute', 'engineering simulation', 'dr caicedo area specialization structural engineering emphasis structural dynamic numerical experimental area structural dynamic', 'specialization structural', 'network earthquake', 'complete list', 'earthquake engineering institute', 'american society', 'publication visit', 'engineering emphasis', 'numerical experimental', 'dynamic numerical', 'lab website', 'jr network', 'emphasis structural', 'society engineering', 'society experimental mechanic american society engineering educator complete list project publication visit structural dynamic intelligent infrastructure lab website', 'society civil', 'educator complete', 'experimental mechanic', 'member american'}

Professor Name: Carrilho, Leo
Skill Set: {'thermal hydraulic', 'using fea cfd method', 'cfd method', 'leo carrilho', 'phenomenon structural', 'dr leo', 'pwr nuclear', 'nuclear fuel', 'using fea', 'modeling simulation', 'carrilho modeling', 'fea cfd', 'fuel core', 'structural thermal', 'process phenomenon', 'design pwr', 'dr leo carrilho modeling simulation process phenomenon structural thermal hydraulic design pwr nuclear fuel core component', 'simulation process', 'hydraulic design', 'core component'}

Professor Name: Carver, Wayne
Skill Set: {'extracellular matrix', 'seen alcohol', 'used examine', 'associated hypertension', 'non muscle', 'interconnected supported', 'endothelial cell others cell interconnected supported elaborate extracellular matrix change density organization extracellular matrix known affect cardiovascular performance correlated heart disease instance', 'hypertension myocardial', 'supported elaborate', 'heart produced', 'cardiovascular disease', 'increased accumulation', 'endothelial cell', 'stiffer myocardium', 'fibroblast behavior', 'contractile muscle', 'density organization', 'school medicine', 'particularly interested', 'room garner', 'ferry road', 'infarction increased', 'fibrosis commonly', 'fibrosis result', 'performance correlated', 'myocardial fibrosis', 'fibroblast cardiovascular', 'accumulation extracellular', 'cell others', 'animal model', 'model used', 'regulated heart', 'culture animal', 'seen fibrosis', 'excessive accumulation extracellular matrix fibrosis commonly associated hypertension myocardial infarction increased accumulation extracellular matrix seen fibrosis result stiffer myocardium', 'matrix known', 'fibrosis cell', 'chronic exposure', 'fibroblast dr', 'regulation fibroblast', 'heart particularly', 'school medicine bldg room garner ferry road columbia', 'known affect', 'focused understanding', 'largely cardiac', 'including fibroblast', 'gene expression', 'heart function', 'cell interconnected', 'excessive accumulation', 'wall heart', 'alcohol seen', 'abuse result', 'matrix change', 'alcohol abuse', 'carver lab', 'road columbia', 'understanding fibroblast', 'cardiovascular performance', 'disease instance', 'expression regulated', 'cell culture', 'cardiac fibroblast', 'exposure alcohol', 'composed contractile', 'sc ventricular', 'heart disease', 'elaborate extracellular', 'ventricular wall', 'garner ferry', 'others cell', 'bldg room', 'sc ventricular wall heart composed contractile muscle cell non muscle cell type including fibroblast', 'change density', 'result myocardial', 'examine regulation', 'commonly associated', 'alters heart', 'organization extracellular', 'behavior gene', 'matrix fibrosis', 'type including', 'affect cardiovascular', 'dr carver', 'interested chronic', 'function extracellular', 'lab focused', 'cell type', 'correlated heart', 'matrix heart', 'myocardial infarction', 'alters heart function extracellular matrix heart produced largely cardiac fibroblast dr carver lab focused understanding fibroblast behavior gene expression regulated heart particularly interested chronic exposure alcohol seen alcohol abuse result myocardial fibrosis cell culture animal model used examine regulation fibroblast cardiovascular disease', 'heart composed', 'cell non', 'produced largely', 'muscle cell', 'medicine bldg', 'result stiffer', 'matrix seen'}

Professor Name: Chandrashekhar, MVS
Skill Set: {'sic', 'chemistry', 'epitaxy', 'plasmonics', 'graphene'}

Professor Name: Chao, Yuh J.
Skill Set: {'chen bingquan', 'material structure', 'stress distortion', 'guian', 'theoretical experimental', 'welding modeling residual stress distortion', 'study failure', 'cylindrical shell', 'quasi perfect', 'experimental numerical', 'international journal', 'fuel cell', 'dukaifan', 'welding modeling', 'chao yuh', 'fatigue material', 'mechanic material', 'asme journal pressure vessel technology', 'perfect cylindrical', 'dr chao theoretical experimental study failure', 'cell system', 'biomechanics nanomechanics', 'yupeng qian', 'hao peng', 'material characterization', 'durability pem fuel cell system cao', 'yuh', 'pressure vessel', 'constraint assessment specimen tested uniaxial biaxial loading condition', 'dr chao', 'xiangtao', 'biaxial loading', 'tested uniaxial', 'journal pressure', 'combined experimental', 'failure criterion', 'vessel technology', 'characterization failure', 'yinbiao niffenegger', 'pem fuel', 'loading condition', 'markus chao', 'durability pem', 'system cao', 'wang bo', 'impact mechanic', 'bi xiangju', 'buckling quasi', 'compression combined', 'solid structure', 'fracture fatigue material structure', 'specimen tested', 'asme journal', 'journal solid', 'residual stress', 'chao theoretical', 'experimental study', 'uniaxial biaxial', 'buckling quasi perfect cylindrical shell axial compression combined experimental numerical investigation', 'international journal solid structure', 'shell axial', 'vol', 'axial compression', 'zhu shiyang', 'constraint assessment', 'numerical investigation', 'fracture fatigue', 'june wang', 'june wang bo', 'assessment specimen', 'impact mechanic material characterization failure criterion', 'modeling residual'}

Professor Name: Chaudhry, Hanif
Skill Set: {'water resource engineering', 'hydraulic transient', 'resource engineering', 'hydraulic engineering', 'water resource'}

Professor Name: Chen, Fanglin (Frank)
Skill Set: {'solid oxide electrolysis cell', 'synthesis characterization', 'solid oxide', 'electronic conducting', 'dr chen', 'gas separation', 'characterization material', 'conversion storage', 'dr chen synthesis characterization material energy conversion storage', 'catalysis electrocatalysis', 'fuel cell', 'solid oxide fuel cell', 'electrolysis cell', 'conducting ceramic', 'oxide electrolysis', 'ionic electronic', 'oxide fuel', 'separation membrane', 'chen synthesis', 'structure property', 'property relationship', 'structure property relationship', 'gas separation membrane', 'energy conversion', 'material energy', 'composition', 'ionic electronic conducting ceramic'}

Professor Name: Chen, Yinchao
Skill Set: {'sc', 'swearingenroom main', 'swearingenroom main streetcolumbia', 'main streetcolumbia'}

Professor Name: Chen, Yuche
Skill Set: {'data sensing', 'sensing analytics', 'aiaugmented simulation', 'data sensing analytics', 'emerging technology'}

Professor Name: Cheng, Xu
Skill Set: {'technology kit', 'nuclear safety', 'ha published', 'system kit', 'cheng ha', 'innovative nuclear', 'kit ha', 'field nuclear', 'director division innovative nuclear system kit ha supervised supervising phd student kit sjtu give lecture undergraduate master student ha published paper main field nuclear thermal hydraulics nuclear safety', 'sjtu china', 'experience currently', 'master student', 'phd student', 'kit germany', 'nuclear system', 'thermal hydraulics', 'supervising phd', 'institute technology', 'published paper', 'jiao tong', 'lecture undergraduate', 'hydraulics nuclear', 'ha year', 'prof cheng ha year experience year teaching experience currently professor karlsruhe institute technology kit germany shanghai jiao tong sjtu china', 'germany shanghai', 'year teaching', 'tong sjtu', 'paper main', 'director division', 'ha supervised', 'student kit', 'student ha', 'professor karlsruhe', 'give lecture', 'experience year', 'undergraduate master', 'year experience', 'teaching experience', 'karlsruhe institute', 'currently professor', 'supervised supervising', 'prof cheng', 'main field', 'nuclear thermal', 'shanghai jiao', 'division innovative', 'sjtu give', 'kit sjtu'}

Professor Name: Cole, Casey
Skill Set: {'bioinformatics', 'computational biology'}

Professor Name: Coman, Paul
Skill Set: {'thermodynamics', 'liion safety'}

Professor Name: Crichigno, Jorge
Skill Set: {'programmable switch', 'science dmz', 'iot security'}

Professor Name: Cui, Taixing
Skill Set: {'fourth military', 'military medical', 'garner ferry', 'school medicine', 'fourth military medical', 'sc', 'molecular medicine', 'bldg room', 'japan', 'room garner', 'ferry road', 'medicine bldg', 'ehime school medicine', 'road columbia', 'r china', 'school medicine bldg room garner ferry road columbia', 'ehime school'}

Professor Name: De Backer, Wout
Skill Set: {'fused deposition modeling', 'printing', 'additive manufacturing', 'deposition modeling', 'fused deposition'}

Professor Name: Deng, Xiaomin
Skill Set: {'process simulation', 'molecular dynamic simulation', 'finite element', 'modeling simulation', 'manufacturing process simulation', 'molecular dynamic', 'manufacturing process', 'dynamic simulation', 'solid mechanic'}

Professor Name: Devereux, Emily
Skill Set: {'based principal', 'dependency theory', 'past year', 'ranking perceived', 'foundation agency', 'agent resource', 'resource dependency', 'bias funding', 'community economic', 'capacity reputation', 'administrator year', 'reputation goal', 'year administration', 'based administrative', 'review process', 'funding sponsor', 'past year administration leadership experience ha concentrated strategic growth', 'senior administrator year administration community economic development', 'concentrated strategic', 'strategic planning', 'signal competitiveness', 'strategic growth', 'leadership experience', 'experience ha', 'funding review', 'perceived bias', 'administration leadership', 'focused national', 'national ranking', 'process based', 'administrative capacity', 'administration focused', 'federal foundation', 'administration focused national ranking perceived bias funding review process based administrative capacity reputation goal identify contributes perceived bias federal foundation agency review enables institution signal competitiveness funding sponsor based principal agent resource dependency theory', 'agency review', 'competitiveness funding', 'ha concentrated', 'review enables', 'sponsor based', 'identify contributes', 'senior administrator', 'development', 'institution signal', 'administration community', 'economic development', 'principal agent', 'student growth', 'bias federal', 'contributes perceived', 'goal identify', 'enables institution'}

Professor Name: Dillon, Anthony L
Skill Set: {'technology clinical', 'partnership relationship', 'related educational', 'undergraduate graduate', 'brings numerous', 'dillon director', 'within iit', 'sector related', 'innovation assembly street', 'suite', 'director internship', 'department integrated', 'integrated information', 'business partnership', 'innovation assembly', 'educational technology', 'numerous business', 'private public', 'assembly street', 'information technology', 'worked within', 'program within', 'technology brings', 'graduate program', 'mhit program', 'within private', 'instructor undergraduate', 'sc dillon director internship department integrated information technology clinical instructor undergraduate graduate program within iit mhit program ha worked within private public sector related educational technology brings numerous business partnership relationship', 'sc dillon', 'public sector', 'internship department', 'clinical instructor', 'ha worked', 'iit mhit', 'program ha'}

Professor Name: Dougal, Roger A.
Skill Set: {'hybrid power', 'sponsorship office', 'better realize', 'prof dougal', 'bed software', 'energy system', 'dougal ha', 'several post doctoral scholar faculty', 'electrochemical power', 'related new', 'main street', 'capacity oversees', 'range associated', 'electric system', 'comprehensive simulation virtual prototyping environment multidisciplinary dynamic system environment applied study electric system navy ship', 'grape joint', 'encompasses wide', 'sc dougal lead power energy system group', 'sponsored industry', 'electronics also', 'engineering department', 'electronics utility', 'grid since', 'dougal currently', 'dozen graduate', 'currently supervises', 'graduate student', 'associated technology', 'environment applied', 'site director', 'electrochemical power source', 'number engineering', 'power generation', 'advanced power', 'department dr', 'navy ship', 'dynamic system', 'power electronic', 'control dr dougal currently supervises dozen graduate student', 'sponsorship office naval', 'dr dougal', 'dougal member', 'member school', 'arkansas seek', 'grid better', 'electric ship', 'nsf sponsored', 'street columbia', 'school prof', 'several post', 'center grid', 'wide range', 'multidisciplinary dynamic', 'technology across', 'lead power', 'simulation virtual', 'coordination activity member school prof dougal also site director new nsf sponsored industry cooperative center grid connected advanced power electronic system grape joint project usc arkansas seek insert greater level advanced power electronics utility power grid better realize smart grid since', 'member board', 'consortium esrdc', 'post doctoral', 'coordination activity', 'industry cooperative', 'seek insert', 'oversees usc', 'power energy', 'insert greater', 'system environment', 'joint project', 'number undergraduate researcher', 'power grid', 'project usc', 'also encompasses', 'usc activity', 'new power', 'test bed', 'supervises dozen', 'connected advanced', 'board director', 'smart grid', 'also site', 'director new', 'prof dougal ha overseen development virtual test bed software', 'undergraduate researcher', 'system group', 'principally focus', 'electronic system', 'control dr', 'usc arkansas', 'virtual prototyping', 'capacity oversees usc activity related new power generation', 'greater level', 'level advanced', 'power source', 'distribution technology', 'applied study', 'environment multidisciplinary', 'power electronics', 'technology ship', 'across number', 'swearingenroom main', 'new nsf', 'scholar faculty', 'principally focus power electronics also encompasses wide range associated technology across number engineering department dr dougal member board director electric ship development consortium esrdc', 'activity related', 'activity member', 'prototyping environment', 'focus power', 'processing', 'virtual test', 'development virtual', 'doctoral scholar', 'ha overseen', 'director electric', 'system navy', 'realize smart', 'development consortium', 'sc dougal', 'distribution technology ship', 'swearingenroom main street columbia', 'hybrid power source', 'comprehensive simulation', 'number undergraduate', 'office naval', 'cooperative center', 'overseen development', 'system grape', 'utility power', 'study electric', 'grid connected', 'dougal lead', 'ship development', 'dougal also'}

Professor Name: Downey, Austin
Skill Set: {'health monitoring', 'realtime system', 'structural health', 'structural health monitoring'}

Professor Name: Dryer, Frederick  L.
Skill Set: {'chemical kinetic', 'chemical kinetics relevant space air breathing propulsion', 'emission interaction', 'gas phase', 'alternative fuel', 'incineration', 'conversion related', 'space air', 'emission generation', 'transportation stationary', 'application driven', 'chemical kinetic property', 'property non', 'relevant space', 'combustion property', 'main street', 'derived fuel', 'combustion related pollutant mitigation', 'ha year', 'phase interaction', 'effect stationary', 'gravity environment', 'pollutant concern', 'related issue', 'carbon cycle', 'material emission', 'concern fire', 'interaction related', 'physical chemistry', 'sc dryer ha year application driven fundamental experience thermal science heat transfer', 'emission internal', 'power generation', 'including gasoline', 'reduction net', 'energy security', 'interaction including', 'emission generation abatement petroleum derived fuel', 'internal combustion', 'aerosol particulate', 'phase gas', 'safety related', 'kinetics fuel', 'dryer current', 'abatement petroleum', 'cycle emission', 'particle burning', 'sc dryer', 'hazardous waste', 'including chlorine', 'catalyst material', 'environment solid', 'heavy fuel oil combustion property non petroleum derived alternative fuel', 'energy conversion', 'dryer ha', 'fuel hazardous', 'kinetics relevant', 'petroleum derived', 'aerosol particulate emission interaction including chlorine', 'driven fundamental', 'burning phenomenon', 'heat transfer', 'air breathing', 'fire safety', 'security reduction', 'component effect', 'chemical kinetics', 'ability address energy security reduction net carbon cycle emission well pollutant concern fire safety related issue earth micro gravity environment solid phase gas phase interaction related particle burning phenomenon nano catalyst material emission internal combustion engine', 'nano catalyst', 'stationary power', 'science heat', 'thermal science', 'nitrogen oxide', 'micro gravity', 'production', 'year application', 'fundamental experience', 'generation abatement', 'combustion', 'stationary energy', 'sulfur ash metal component effect stationary energy conversion', 'sulfur ash', 'issue earth', 'including hydrocarbon', 'fluid dynamic', 'chemical processing', 'ability address', 'dr dryer', 'chemistry chemical', 'fuel oil', 'non petroleum', 'derived alternative', 'solid phase', 'particulate emission', 'diesel', 'kinetic property', 'related ignition', 'combustion related', 'net carbon', 'gas turbine', 'breathing propulsion', 'combustion engine', 'ash metal', 'related particle', 'fire safety dr dryer current chemistry chemical kinetics fuel hazardous waste material related ignition', 'oil combustion', 'experience thermal', 'emission well', 'fossil renewable', 'renewable energy', 'metal component', 'related pollutant', 'safety dr', 'related ground', 'waste material', 'earth micro', 'rm carolinacolumbia', 'well pollutant', 'pollutant mitigation', 'ground transportation', 'heavy fuel', 'current chemistry', 'material related', 'fossil renewable energy conversion related ground transportation stationary power generation', 'phenomenon nano', 'address energy'}

Professor Name: Eastman, Caroline M.
Skill Set: {'search interface', 'professor eastman focus effective efficient algorithm information retrieval want information', 'want information', 'correct find', 'database security', 'efficient algorithm', 'easily quickly', 'eastman focus', 'file organization', 'professor eastman', 'focus effective', 'efficient file organization search interface user interaction ha also working colleague problem database security pi co director professor bowles nsf experience undergraduate site', 'interaction ha', 'ha also', 'director professor', 'easily quickly find useful correct find work area ha addressed broad spectrum problem', 'user interaction', 'bowles nsf', 'useful correct', 'work area', 'area ha', 'colleague problem', 'nsf experience', 'experience undergraduate', 'organization search', 'retrieval want', 'find useful', 'interface user', 'effective efficient', 'find work', 'pi co', 'ha addressed', 'security pi', 'spectrum problem', 'efficient file', 'undergraduate site', 'professor bowles', 'information retrieval', 'addressed broad', 'problem database', 'working colleague', 'algorithm information', 'broad spectrum', 'quickly find', 'co director', 'also working'}

Professor Name: Eberth, John F.
Skill Set: {'biomechanics', 'biomedical engineering', 'mechanobiology', 'cardiovascular'}

Professor Name: Ebner, Armin D.
Skill Set: {'found medical', 'fine ferromagnetic', 'emphasis use', 'today major', 'consists use', 'catholic chile', 'ebner major', 'gradient magnetic', 'assist magnetic', 'field separation', 'magnetic drug', 'ebner major application magnetic field separation', 'size low', 'fluid medium', 'separation hgms', 'field become', 'used assist', 'major emphasis', 'particular emphasis use high gradient magnetic separation hgms principle technique consists use fine ferromagnetic element result application external magnetic field become energized create magnetic gradient large enough collect particle small size low magnetism collection otherwise realized today major emphasis potential hgms found medical area technique used assist magnetic drug targeting cancer treatment', 'hgms found', 'particle small', 'use fine', 'collect particle', 'medical area', 'application external', 'result application', 'magnetic particle', 'drug targeting', 'collection manipulation magnetic particle fluid medium', 'collection otherwise', 'technique used', 'embolization control', 'external magnetic', 'technique consists', 'control well', 'magnetic field', 'high gradient', 'principle technique', 'particle fluid', 'embolization control well rapid detoxification', 'chile', 'particular emphasis', 'magnetic separation', 'cancer treatment', 'small size', 'hgms principle', 'potential hgms', 'otherwise realized', 'manipulation magnetic', 'area technique', 'collection manipulation', 'ph', 'restenosis', 'enough collect', 'gradient large', 'magnetism collection', 'energized create', 'element result', 'application magnetic', 'use high', 'ferromagnetic element', 'low magnetism', 'emphasis potential', 'well rapid', 'major application', 'become energized', 'large enough', 'realized today', 'targeting cancer', 'rapid detoxification', 'magnetic gradient', 'create magnetic'}

Professor Name: Eslambolchi Moghadam, Sara
Skill Set: {'wound healing', 'medicinal plant', 'natural product', 'drug discovery'}

Professor Name: Fan, Daping
Skill Set: {'including hypoxia', 'developing strategy', 'response form', 'deposit disease', 'network developing', 'killer united', 'dendritic cell', 'killer united state many developed country atherosclerosis lipid deposit disease chronic inflammatory condition macrophage primary cholesterol sink well dominant inflammation machine atherogenesis macrophage foam cell formation inflammatory response form vicious cycle promote atherogenesis goal laboratory develop strategy disrupt vicious cycle promote regression atherosclerotic plaque restoring macrophage cholesterol homeostasis controlling macrophage inflammation myeloid cell major cell component tumor microenvironment tme', 'tme mainly', 'improve cancer', 'feature tme', 'restoring macrophage', 'cell formation', 'toll like', 'mainly myeloid', 'unusual metabolic', 'cancer treatment', 'cell component', 'understanding signal', 'machine atherogenesis', 'malignant cell', 'network tune', 'due unique feature tme including hypoxia', 'controlling macrophage', 'sc rupture', 'resulting thrombosis', 'atherogenesis goal', 'formation inflammatory', 'manipulate improve', 'many developed', 'metabolic profile', 'school medicine', 'macrophage cholesterol', 'room garner', 'ferry road', 'derived suppressor', 'like receptor', 'tumor associated', 'thrombosis direct', 'signaling network', 'goal laboratory', 'function myeloid', 'tumor associated macrophage tam advanced tumor', 'whose interaction', 'sc rupture vulnerable atherosclerotic plaque resulting thrombosis direct cause heart attack stroke', 'major cell', 'tumor myeloid', 'form vicious', 'favor cancer', 'sink well', 'suppressor cell', 'tumor microenvironment', 'cholesterol sink', 'atherosclerotic plaque', 'school medicine bldg room garner ferry road columbia', 'united state', 'macrophage foam', 'atherosclerosis lipid', 'focused understanding', 'determines aggressiveness', 'cytokine chemokines', 'cholesterol homeostasis', 'aggressiveness tumor', 'macrophage inflammation', 'promote regression', 'state many', 'cell tme', 'component tumor', 'myeloid derived', 'associated macrophage', 'tam advanced', 'cancer progression', 'plaque resulting', 'complex signaling network tune dynamic function myeloid cell favor cancer progression focused understanding signal network developing strategy manipulate improve cancer treatment', 'attack stroke', 'vicious cycle', 'cell favor', 'inflammation myeloid', 'strategy disrupt', 'inflammatory response', 'microenvironment tme', 'cell major', 'abundance cytokine chemokines toll like receptor tlr ligand', 'road columbia', 'lipid deposit', 'dynamic function', 'regression atherosclerotic', 'heart attack', 'strategy manipulate', 'promote atherogenesis', 'laboratory develop', 'direct cause', 'homeostasis controlling', 'country atherosclerosis', 'garner ferry', 'foam cell', 'bldg room', 'atherogenesis macrophage', 'interaction malignant', 'cause heart', 'receptor tlr', 'macrophage primary', 'vulnerable atherosclerotic', 'plaque restoring', 'inflammatory condition', 'develop strategy', 'developed country', 'unique feature', 'disease chronic', 'complex signaling', 'tune dynamic', 'signal network', 'advanced tumor', 'macrophage tam', 'whose interaction malignant cell determines aggressiveness tumor myeloid cell tme mainly myeloid derived suppressor cell mdscs', 'chemokines toll', 'cell mdscs', 'disrupt vicious', 'cycle promote', 'cell determines', 'tlr ligand', 'dominant inflammation', 'primary cholesterol', 'rupture vulnerable', 'well dominant', 'inflammation machine', 'myeloid cell', 'condition macrophage', 'chronic inflammatory', 'abundance cytokine', 'due unique', 'medicine bldg', 'tme including', 'unusual metabolic profile', 'progression focused'}

Professor Name: Farkas, Csilla
Skill Set: {'information security'}

'''

In [ ]:
available_proposals = '''

Available Proposals:

Title: Advanced Technological Education
Skills Required: {'industry', 'higher education', 'engineering', 'professional development', 'development', 'science engineering', 'material', 'economic development'}

Title: Environmental Convergence Opportunities in Chemical, Bioengineering, Environmental, and Transport Systems
Skills Required: {'engineering', 'science engineering', 'environmental engineering', 'chemical'}

Title: Collaborative Research in Computational Neuroscience
Skills Required: {'theoretical foundation', 'computer science'}

Title: National Artificial Intelligence (AI) Research Institutes
Skills Required: {'industry', 'continued advancement', 'economic impact', 'development', 'security', 'machine learning', 'learning', 'ai', 'artificial intelligence'}

Title: Computer and Information Science and Engineering (CISE): Core Programs
Skills Required: {'science engineering', 'project develop', 'engineering', 'information science', 'communication', 'computer information'}

Title: Division of Materials Research: Topical Materials Research Programs: Biomaterials (BMAT), Condensed Matter Physics (CMP), Metals and Metallic Nanostructures (MMN), Polymers (POL) (DMR-TMRP BMAT, CMP, MMN, POL) (nsf20589) | NSF - National Science Foundation
Skills Required: {'property', 'industry', 'design synthesis', 'polymer', 'physic', 'engineering', 'biomaterials', 'novel design', 'national science', 'united state', 'development', 'material unique', 'control', 'science foundation', 'fundamental understanding', 'lead new', 'synthesis characterization', 'material', 'processing', 'synthesis', 'discovery'}

Title: Division of Materials Research: Topical Materials Research Programs: Ceramics (CER), Electronic and Photonic Materials (EPM), Solid State and Materials Chemistry (SSMC)
Skills Required: {'property', 'industry', 'design synthesis', 'engineering', 'novel design', 'united state', 'development', 'material unique', 'control', 'chemistry', 'fundamental understanding', 'lead new', 'synthesis characterization', 'material', 'processing', 'synthesis', 'solid state', 'discovery', 'electronic photonic'}

Title: Re-entry to Active Research Program
Skills Required: {'chemistry', 'chemical'}

Title: Research Experiences for Teachers (RET) in Engineering and Computer Science (nsf20584) | NSF - National Science Foundation
Skills Required: {'industry', 'revolve around', 'engineering', 'national science', 'computer information', 'directorate engineering', 'science technology', 'science foundation', 'science engineering', 'experience new', 'material', 'area', 'information science', 'computer science'}

Title: Condensed Matter and Materials Theory (CMMT) (nsf20582) | NSF - National Science Foundation
Skills Required: {'national science', 'science foundation', 'electronic photonic', 'broad spectrum', 'polymer', 'molecular dynamic', 'material soft', 'material state', 'soft material', 'material', 'statistical mechanic', 'physic', 'biomaterials', 'polymeric material', 'dynamic', 'area', 'learning', 'material system', 'solid state', 'property', 'development', 'chemistry', 'fundamental understanding', 'soft matter', 'method used', 'machine learning', 'modeling'}

Title: Division of Chemistry: Disciplinary Research Programs
Skills Required: {'imaging', 'computational method', 'catalysis', 'chemistry', 'dynamic', 'synthesis', 'chemical'}

Title: Joint DMS/NIGMS Initiative to Support Research at the Interface of the Biological and Mathematical Sciences
Skills Required: {'support interface', 'statistic', 'biomedical', 'health', 'national science', 'science foundation', 'biomedical science', 'national institute'}

Title: Centers for Chemical Innovation
Skills Required: {'chemical', 'communication'}

Title: NSF Program on Fairness in Artificial Intelligence in Collaboration with Amazon
Skills Required: {'ha long', 'algorithm', 'health', 'security', 'ai system', 'decision support', 'support system', 'machine learning', 'learning', 'ai', 'artificial intelligence'}

Title: Opportunities for Promoting Understanding through Synthesis
Skills Required: {'synthesis'}

Title: Cyber-Physical Systems
Skills Required: {'personalized medicine', 'automation', 'civil', 'energy', 'security', 'machine learning', 'learning', 'range application', 'artificial intelligence'}

Title: Hydrologic Sciences (HS)
Skills Required: {'biological process', 'engineering', 'experimental theoretical', 'energy', 'science engineering', 'modeling', 'synthesis', 'including limited', 'energy transport', 'chemical'}

Title: Spectrum Innovation Initiative: National Center for Wireless Spectrum Research
Skills Required: {'system application', 'communication', 'thing iot', 'high speed', 'application including', 'wireless communication', 'internet thing'}

Title: Gen-4 Engineering Research Centers
Skills Required: {'including engineering', 'engineering', 'development'}

Title: Future Manufacturing
Skills Required: {'logistics', 'machine design', 'production', 'business', 'material', 'chemical'}

Title: Expeditions in Computing (Expeditions) (nsf20544) | NSF - National Science Foundation
Skills Required: {'science foundation', 'science engineering', 'engineering', 'information science', 'national science', 'computer information'}

Title: Reproducible Cells and Organoids via Directed-Differentiation Encoding
Skills Required: {'cell cell', 'foundation nsf', 'national science', 'control', 'science foundation', 'fundamental understanding', 'product', 'develop strategy', 'dynamic', 'cell type', 'chemical'}

Title: NSF-Simons Research Collaborations on the Mathematical and Scientific Foundations of Deep Learning
Skills Required: {'deep learning', 'graduate student', 'principal investigator', 'engineering', 'activity focused', 'national science', 'computer information', 'science foundation', 'science engineering', 'area', 'learning', 'information science', 'artificial intelligence', 'new york'}

Title: Computer Science for All
Skills Required: {'foundation nsf', 'professional development', 'national science', 'development', 'science foundation', 'material', 'teaching', 'computer science'}

Title: National Robotics Initiative 2.0: Ubiquitous Collaborative Robots
Skills Required: {'united state', 'development', 'robotics'}

Title: Environmental Convergence Opportunities in Chemical, Bioengineering, Environmental, and Transport Systems
Skills Required: {'science engineering', 'engineering', 'environmental engineering', 'chemical'}

Title: Navigating the New Arctic (NNA) (nsf20514) | NSF - National Science Foundation
Skills Required: {'foundation nsf', 'engineering', 'program director', 'national science', 'science foundation', 'science engineering', 'area future', 'area'}

Title: Understanding the Rules of Life: Microbiome Theory and Mechanisms (URoL:MTM) (nsf20513) | NSF - National Science Foundation
Skills Required: {'foundation nsf', 'engineering', 'program director', 'national science', 'science foundation', 'science engineering', 'area future', 'area'}

Title: Addressing Systems Challenges through Engineering Teams (ASCENT) (nsf20511) | NSF - National Science Foundation
Skills Required: {'system application', 'big data', 'communication', 'engineering', 'national science', 'energy', 'control', 'science foundation', 'security', 'cyber system', 'machine learning', 'synthesis', 'area', 'learning', 'fuel', 'information science', 'artificial intelligence'}

Title: NSF Engineering - UKRI Engineering and Physical Sciences Research Council Lead Agency Opportunity
Skills Required: {'division electrical', 'communication', 'review process', 'engineering', 'civil', 'national science', 'directorate engineering', 'science foundation', 'cyber system', 'chemical'}

Title: Transitions to Excellence in Molecular and Cellular Biosciences Research (Transitions) (nsf20505) | NSF - National Science Foundation
Skills Required: {'retention', 'professional development', 'track', 'national science', 'development', 'science foundation', 'full professor', 'area', 'year'}

Title: National Artificial Intelligence (AI) Research Institutes
Skills Required: {'industry', 'continued advancement', 'economic impact', 'development', 'security', 'machine learning', 'learning', 'ai', 'artificial intelligence'}

Title: NSF/CASIS Collaboration on Transport Phenomena Research on the International Space Station (ISS) to Benefit Life on Earth
Skills Required: {'thermal transport', 'transport process', 'foundation nsf', 'engineering', 'national science', 'science foundation', 'combustion', 'transport phenomenon', 'fluid dynamic', 'dynamic', 'chemical'}

Title: NSF/CASIS Collaboration on Tissue Engineering and Mechanobiology on the International Space Station (ISS) to Benefit Life on Earth (nsf20500) | NSF - National Science Foundation
Skills Required: {'foundation nsf', 'engineering', 'civil', 'national science', 'mechanobiology', 'science foundation', 'tissue engineering', 'chemical'}

Title: IUSE / Professional Formation of Engineers:
Skills Required: {'engineering material', 'engineering education', 'engineering department', 'capstone design', 'first year', 'engineering', 'professional engineer', 'material', 'learning', 'professional practice', 'year'}

Title: Formal Methods in the Field
Skills Required: {'system application', 'embedded system', 'engineering', 'computer information', 'science engineering', 'machine learning', 'modeling', 'synthesis', 'area', 'learning', 'information science', 'year'}

Title: EMERGING FRONTIERS IN RESEARCH AND INNOVATION
Skills Required: {'area', 'engineering', 'directorate engineering'}

Title: NSF/DOE Partnership in Basic Plasma Science and Engineering
Skills Required: {'property', 'physic', 'engineering', 'material science', 'applied mathematics', 'science engineering', 'material', 'plasma'}

Title: Computer and Information Science and Engineering (CISE): Core Programs
Skills Required: {'science engineering', 'project develop', 'engineering', 'information science', 'communication', 'computer information'}

Title: Division of
Skills Required: {'imaging', 'computational method', 'catalysis', 'chemistry', 'dynamic', 'synthesis', 'chemical'}

Title: Methodology, Measurement, and Statistics
Skills Required: {'mm', 'statistic', 'development'}

Title: Secure and Trustworthy Cyberspace Frontiers
Skills Required: {'engineering', 'security', 'cyber system', 'problem involving'}

Title: NSF Program on Fairness in Artificial Intelligence in Collaboration with Amazon (FAI) (nsf19571) | NSF - National Science Foundation
Skills Required: {'ha long', 'health', 'national science', 'science foundation', 'security', 'ai system', 'machine learning', 'learning', 'ai', 'artificial intelligence'}

Title: Cyber-Physical Systems
Skills Required: {'personalized medicine', 'civil', 'energy', 'security', 'range application', 'artificial intelligence'}

Title: National Robotics Initiative 2.0: Ubiquitous Collaborative Robots (NRI-2.0) (nsf19536) | NSF - National Science Foundation
Skills Required: {'national science', 'united state', 'development', 'science foundation', 'robotics'}

Title: Enabling Quantum Leap: Quantum Idea Incubator for Transformational Advances in Quantum Systems (QII - TAQS) (nsf19532) | NSF - National Science Foundation
Skills Required: {'foundation nsf', 'engineering', 'program director', 'national science', 'science foundation', 'science engineering', 'area future', 'area'}

Title: Spectrum Efficiency, Energy Efficiency, and Security (SpecEES):
Skills Required: {'engineering', 'national science', 'directorate engineering', 'computer information', 'energy', 'science foundation', 'science engineering', 'service', 'security', 'information science'}

Title: CNH2: Dynamics of Integrated Socio-Environmental Systems
Skills Required: {'dynamic', 'chemical'}

Title: NSF/CASIS Collaboration on Transport Phenomena Research on the International Space Station (ISS) to Benefit Life on Earth
Skills Required: {'thermal transport', 'transport process', 'foundation nsf', 'engineering', 'national science', 'science foundation', 'combustion', 'transport phenomenon', 'fluid dynamic', 'dynamic', 'chemical'}

Title: Methodology, Measurement, and Statistics (MMS) (nsf19520) | NSF - National Science Foundation
Skills Required: {'statistic', 'national science', 'development', 'science foundation', 'mm', 'production'}

Title: Designing Materials to Revolutionize and Engineer our Future (DMREF) (nsf19516) | NSF - National Science Foundation
Skills Required: {'property', 'system design', 'engineering', 'material science', 'national science', 'development', 'coordination activity', 'science foundation', 'science engineering', 'material', 'advanced material', 'discovery'}

Title: Navigating the New Arctic
Skills Required: {'foundation nsf', 'engineering', 'program director', 'national science', 'science foundation', 'science engineering', 'area future', 'area'}

Title: NSF/CASIS Collaboration on Tissue Engineering and Mechanobiology on the International Space Station (ISS) to Benefit Life on Earth (nsf19509) | NSF - National Science Foundation
Skills Required: {'foundation nsf', 'engineering', 'civil', 'national science', 'mechanobiology', 'science foundation', 'tissue engineering', 'chemical'}

Title: Scalable Parallelism in the Extreme (SPX) (nsf19505) | NSF - National Science Foundation
Skills Required: {'national science', 'science foundation', 'year'}

Title: Big Data Regional Innovation Hubs
Skills Required: {'industry', 'big data', 'engineering', 'united state', 'computer information', 'science engineering', 'information science'}

Title: Formal Methods in the Field
Skills Required: {'system application', 'engineering', 'computer information', 'science engineering', 'machine learning', 'modeling', 'synthesis', 'area', 'learning', 'information science', 'year'}

Title: Infrastructure Innovation for Biological Research
Skills Required: {'multidisciplinary approach'}

Title: Collaborative Research in Computational Neuroscience
Skills Required: {'theoretical foundation', 'computer science'}

Title: Division of Molecular and Cellular Biosciences: Investigator-initiated research projects (MCB) (nsf18585) | NSF - National Science Foundation
Skills Required: {'national science', 'science foundation'}

Title: Secure and Trustworthy Cyberspace
Skills Required: {'engineering', 'security', 'cyber system', 'problem involving'}

Title: Advanced Technological Education
Skills Required: {'industry', 'higher education', 'engineering', 'professional development', 'development', 'science engineering', 'material'}

Title: Information and Intelligent Systems (IIS): Core Programs
Skills Required: {'project develop', 'intelligent system'}

Title: Computer and Network Systems (CNS): Core Programs
Skills Required: {'development novel', 'development'}

Title: Office of Advanced Cyberinfrastructure (OAC): Research Core Program
Skills Required: {'engineering education', 'engineering', 'development', 'science engineering', 'service', 'security', 'area', 'translational', 'discovery', 'computational data'}

Title: Joint DMS/NIGMS Initiative to Support Research at the Interface of the Biological and Mathematical Sciences
Skills Required: {'support interface', 'statistic', 'foundation nsf', 'biomedical', 'health', 'national science', 'science foundation', 'biomedical science', 'national institute', 'nih'}

Title: Division of Physics: Investigator-Initiated Research Projects
Skills Required: {'broad range', 'physic'}

Title: Division of
Skills Required: {'imaging', 'computational method', 'catalysis', 'chemistry', 'dynamic', 'synthesis', 'chemical'}

Title: Smart and Autonomous Systems
Skills Required: {'perception', 'communication', 'actuation', 'smart grid'}

Title: NSF/FDA SCHOLAR-IN-RESIDENCE AT FDA (nsf18556) | NSF - National Science Foundation
Skills Required: {'system directorate', 'graduate student', 'principal investigator', 'foundation nsf', 'engineering', 'health', 'national science', 'computer information', 'directorate engineering', 'science foundation', 'science engineering', 'engineering division', 'material', 'information science', 'computer science'}

Title: Smart and Connected Health
Skills Required: {'public health', 'process modeling', 'foundation nsf', 'biomedical', 'engineering', 'health', 'national science', 'development', 'computer information', 'science foundation', 'science engineering', 'national institute', 'security', 'modeling', 'area', 'aging', 'nih', 'information science'}

Title: Cyber-Physical Systems (CPS) (nsf18538) | NSF - National Science Foundation
Skills Required: {'personalized medicine', 'civil', 'national science', 'energy', 'science foundation', 'security', 'range application', 'artificial intelligence'}

Title: Computer Science for All
Skills Required: {'foundation nsf', 'professional development', 'national science', 'development', 'science foundation', 'material', 'teaching', 'computer science'}

Title: Formal Methods in the Field
Skills Required: {'system application', 'engineering', 'computer information', 'science engineering', 'machine learning', 'modeling', 'synthesis', 'area', 'learning', 'information science', 'year'}

Title: Expeditions in Computing
Skills Required: {'science engineering', 'engineering', 'information science', 'computer information'}

Title: Re-entry to Active Research Program
Skills Required: {'chemical'}

Title: NSF/CASIS Collaboration on Fluid Dynamics and Particulate and Multiphase Processes Research on the International Space Station to Benefit Life on Earth
Skills Required: {'foundation nsf', 'engineering', 'national science', 'science foundation', 'fluid dynamic', 'dynamic', 'chemical'}

Title: National Robotics Initiative 2.0: Ubiquitous Collaborative Robots (NRI-2.0) (nsf18518) | NSF - National Science Foundation
Skills Required: {'national science', 'united state', 'development', 'science foundation', 'robotics'}

Title: Resource Implementations for Data Intensive Research in the Social, Behavioral and Economic Sciences
Skills Required: {'product', 'area'}

Title: NSF/CASIS Collaboration on Tissue Engineering on the International Space Station to Benefit Life on Earth
Skills Required: {'foundation nsf', 'engineering', 'national science', 'science foundation', 'tissue engineering', 'chemical'}

Title: EPSCoR Research Infrastructure Improvement Program: Track-2 Focused EPSCoR Collaborations (RII Track-2 FEC)
Skills Required: {'industry', 'higher education', 'foundation nsf', 'national science', 'development', 'science foundation', 'program level'}

Title: Collaborative Research in Computational Neuroscience
Skills Required: {'theoretical foundation', 'computer science'}

Title: Condensed Matter and Materials Theory
Skills Required: {'electronic photonic', 'broad spectrum', 'polymer', 'molecular dynamic', 'material soft', 'material state', 'soft material', 'material', 'statistical mechanic', 'physic', 'biomaterials', 'polymeric material', 'dynamic', 'area', 'learning', 'material system', 'solid state', 'property', 'development', 'chemistry', 'fundamental understanding', 'soft matter', 'method used', 'machine learning', 'modeling'}

Title: Spectrum Efficiency, Energy Efficiency, and Security (SpecEES): Enabling Spectrum for All
Skills Required: {'engineering', 'national science', 'directorate engineering', 'computer information', 'energy', 'science foundation', 'science engineering', 'service', 'security', 'information science'}

Title: Scalable Parallelism in the Extreme
Skills Required: {'year'}

Title: Division of Molecular and Cellular Biosciences: Investigator-Initiated Research Projects
Skills Required: {'general'}

Title: Division of Materials Research: Topical Materials Research Programs
Skills Required: {'property', 'industry', 'design synthesis', 'engineering', 'novel design', 'united state', 'development', 'material unique', 'control', 'fundamental understanding', 'lead new', 'synthesis characterization', 'material', 'processing', 'synthesis', 'discovery'}

Title: Emerging Frontiers in Research and Innovation 2018
Skills Required: {'area', 'engineering', 'directorate engineering'}

Title: Secure and Trustworthy Cyberspace
Skills Required: {'engineering', 'security', 'cyber system', 'problem involving'}

Title: Research Experiences for Teachers (RET) in Engineering and Computer Science
Skills Required: {'revolve around', 'graduate student', 'engineering', 'computer information', 'directorate engineering', 'science technology', 'science engineering', 'experience new', 'area', 'information science', 'computer science'}

Title: Information and Intelligent Systems (IIS): Core Programs
Skills Required: {'project develop', 'intelligent system'}

Title: Joint DMS/NIGMS Initiative to Support Research at the Interface of the Biological and Mathematical Sciences
Skills Required: {'support interface', 'statistic', 'foundation nsf', 'biomedical', 'health', 'national science', 'science foundation', 'biomedical science', 'national institute', 'nih'}

Title: Advanced Technological Education (ATE) (nsf17568) | NSF - National Science Foundation
Skills Required: {'industry', 'engineering', 'professional development', 'national science', 'development', 'science foundation', 'science engineering', 'material'}

Title: Geophysics
Skills Required: {'property', 'physic', 'wave propagation', 'theoretical experimental', 'experimental study', 'composition', 'material'}

Title: Mathematical Sciences Research Institutes (nsf17553) | NSF - National Science Foundation
Skills Required: {'statistic', 'national science', 'united state', 'science foundation', 'discovery'}

Title: Ideas Lab: Practical Fully-Connected Quantum Computer Challenge (PFCQC)
Skills Required: {'processing', 'information processing', 'quantum computing', 'physic'}

Title: Robert Noyce Teacher Scholarship Program (nsf17541) | NSF - National Science Foundation
Skills Required: {'including engineering', 'engineering', 'track', 'national science', 'science technology', 'science foundation', 'teaching', 'computer science'}

Title: Building Community and Capacity in Data Intensive Research in Education
Skills Required: {'engineering', 'engineering activity', 'creating new', 'science engineering', 'area'}

Title: Innovations at the Nexus of Food, Energy and Water Systems
Skills Required: {'energy'}

Title: Computer Science for All
Skills Required: {'foundation nsf', 'professional development', 'national science', 'development', 'science foundation', 'material', 'teaching', 'computer science'}

Title: Integrative Strategies for Understanding Neural and Cognitive Systems
Skills Required: {'engineering', 'science engineering', 'multidisciplinary approach', 'area', 'broad spectrum'}

'''

In [ ]:
import random
import re

def get_to_match(professors_data):
    pattern = r"Professor Name: (.*?)\nSkill Set: ({.*?})"
    matches = re.findall(pattern, professors_data, re.DOTALL)
    if not matches:
        return "To Match:\n\nProfessor Name: N/A\nSkill Set: { }"

    professor_name, skill_set = random.choice(matches)
    to_match_str = f"To Match:\n\nProfessor Name: {professor_name}\nSkill Set: {skill_set}"
    return to_match_str

# Combine everything into a final prompt
to_match = get_to_match(available_professors)

new_prompt = f"""{prompt2}

{to_match}
{available_professors}
{available_proposals}
"""

# Print or use the new_prompt as needed
print(new_prompt)




You are an intelligent recommendation system designed for research collaborations. Your task is to recommend suitable teams and match them with the most appropriate proposals based on their expertise and specific evaluation metrics. Here’s the process:
Input:

1. Professor’s Name and Skill Set: I will provide a professor’s name along with their specific skills.
2. List of Available Professors and Their Skill Sets: You will receive a list of professors and their respective skills.
3. Available Proposals and Their Details: A list of research proposals with detailed descriptions.
Output:

1. Recommended Team: Based on the given professor’s skills and the available professors’ skills, recommend a team.
2. Proposal Matching: Match the recommended team with the most suitable research proposal.

Do not give any other details. Provide multiple outputs in a proper format of Professor Name; Proposal Name; Team Recommended. Give multiple recommendations. Also, give multiple team recommendations

In [ ]:
prompt = '''

You are an intelligent recommendation system designed for research collaborations. Your task is to recommend suitable teams and match them with the most appropriate proposals based on their skill set. Here's the process:

Input:

List of Available Professors and Their Skill Sets: You will receive a list of other professors and their respective skills.
Available Proposals and Their Details: A list of research proposals with detailed descriptions.

Output:

Proposal Matching and Team Recommendation: Provide and Team Recommendation and Match the recommended team with the most suitable research proposal.
The proposed team can have number of professor less than 5.

Do not give any other details. Provide multiple outputs in a proper format of Proposal Name; Team recommended. Give multiple recommendations for each proposal.
Provide the recommendation output in CSV format.


Available Professors:

Professor Name: Agostinelli, Forest
Skill Set: {'search', 'artificial intelligence', 'reinforcement learning', 'bioinformatics', 'deep learning'}

Professor Name: Ahmad, Iftikhar
Skill Set: {'inc ph', 'computer simulation', 'aluminum content', 'semiconductor including', 'high aluminum', 'gallium oxide', 'previous position', 'ph texas', 'boron nitride', 'high power', 'bandgap semiconductor', 'including high', 'power electronic', 'ultra wide', 'novel high', 'universityresearch growth', 'electronic technology', 'boron nitride gallium oxide', 'position senior', 'nitride gallium', 'growth study', 'fabrication novel', 'content algan', 'sensor electronic', 'study ultra', 'fabrication novel high power electronic photonic device', 'inc ph texas tech universityresearch growth study ultra wide bandgap semiconductor including high aluminum content algan', 'computer simulation device', 'senior scientist', 'previous position senior scientist', 'sensor electronic technology', 'wide bandgap', 'simulation device', 'texas tech', 'photonic device', 'electronic photonic', 'tech universityresearch'}

Professor Name: Alexeev, Oleg S.
Skill Set: {'institute catalysis', 'exafs', 'catalysis considerable', 'cluster aggregate', 'used establish', 'gained technique', 'support interface', 'technique along', 'xrd', 'promoter structure', 'objective program', 'structure active site', 'various experimental', 'intermediate associated', 'site catalyst', 'spectroscopic technique', 'ph', 'adsorbed reactant', 'characterize active', 'structure metal cluster support', 'aggregate objective', 'active site', 'tpr', 'including tpd', 'catalyst adsorbed', 'combination various', 'particle size', 'property metal', 'specie actual', 'place sequence', 'adsorbed specie', 'russiacatalytic process surface take place sequence elementary reaction involving adsorbed reactant', 'catalyst surface', 'condition catalysis', 'chemisorption catalytic', 'boreskov institute catalysis', 'russiacatalytic process', 'establish dependence', 'structure active', 'focused fundamental', 'pursued application', 'xps', 'novosibirsk', 'boreskov institute', 'material spectroscopic', 'influence promoter structure catalytic property supported metal cluster aggregate objective program pursued application combination various experimental method characterization catalytic material spectroscopic technique', 'involving adsorbed', 'used characterize', 'influence promoter', 'fundamental understanding', 'structure metal', 'take place', 'product', 'actual condition', 'catalytic property', 'considerable information', 'property supported', 'metal cluster', 'used characterize active site catalyst adsorbed specie actual condition catalysis considerable information gained technique along chemisorption catalytic data used establish dependence catalytic property metal cluster particle size', 'novosibirsk state', 'program pursued', 'uv visible', 'supported metal', 'cluster support', 'hrtem', 'reaction involving', 'understanding structure', 'method characterization', 'surface goal', 'ftir', 'characterization catalytic', 'associated active', 'along chemisorption', 'surface take', 'sequence elementary', 'application combination', 'russiam', 'structure catalytic', 'process surface', 'cluster particle', 'elementary reaction', 'catalytic material', 'dependence catalytic', 'reaction intermediate', 'data used', 'structure metal support interface', 'metal support', 'reaction intermediate associated active site catalyst surface goal focused fundamental understanding structure metal support interface', 'goal focused', 'information gained', 'experimental method', 'catalytic data'}

Professor Name: Ali, Mohammod
Skill Set: {'usc july', 'director national', 'sc ph', 'nsf ipa', 'science foundation', 'dhaka received sc ph degree electrical engineering', 'mohammod ali', 'engineering usc', 'intergovernmental personnel', 'sc mohammod ali professor department electrical engineering usc july', 'canada', 'engineering bangladesh', 'dhaka received', 'assignment division', 'bangladesh engineering', 'foundation nsf', 'prof ali', 'ali received', 'department electrical', 'working program', 'ipa intergovernmental', 'engineering technology', 'sc mohammod', 'british columbia', 'program director', 'electrical electronic', 'respectively victoria', 'electrical engineering', 'received sc', 'ali professor', 'personnel act', 'division electrical', 'national science', 'electronic engineering', 'professor department', 'act assignment', 'main streetcolumbia', 'system directorate', 'swearingenroom main streetcolumbia', 'degree electrical', 'engineering prof', 'communication', 'cyber system', 'ph degree', 'ha working program director national science foundation nsf ipa intergovernmental personnel act assignment division electrical', 'swearingenroom main', 'ha working', 'sc electrical', 'directorate engineering', 'cyber system directorate engineering prof ali received sc electrical electronic engineering bangladesh engineering technology'}

Professor Name: Ammal, Salai C.
Skill Set: {'bharathidasan', 'ammal pdfph', 'cv dr', 'india', 'cv dr ammal pdfph', 'dr ammal'}

Professor Name: Bakos, Jason D.
Skill Set: {'high performance', 'computer architecture', 'performance computing', 'reconfigurable computing', 'heterogeneous computing', 'high performance computing', 'embedded system'}

Professor Name: Banerjee, Sourav
Skill Set: {'ultrasonics', 'acoustic', 'wave propagation', 'biomedical', 'metamaterials'}

Professor Name: Bayat, Mahmoud
Skill Set: {'nonlinear vibration', 'structural health', 'health monitoring', 'probabilistic analysis', 'earthquake eng', 'machine learning', 'structural health monitoring'}

Professor Name: Bayoumi, Abdel-Moez E.
Skill Set: {'various industry', 'predictive maintenance', 'engineering north', 'ha published', 'mechanical aerospace', 'mechanical material', 'ha received', 'sc bayoumi', 'experience currently', 'strong program', 'prediction mechanical', 'north state', 'bayoumi ha', 'engineering washington', 'industry experience', 'ha year', 'center predictive', 'life prediction', 'year teaching', 'actively involved', 'tribology', 'diagnosis prognosis life prediction mechanical system', 'relation professor', 'national science foundation various industry ha published journal conference paper', 'monitoring system', 'condition based maintenance', 'journal conference', 'sc bayoumi ha year teaching', 'state ha', 'director center', 'professor mechanical', 'project manager hewlett packard company', 'condition based', 'associate dean corporate relation professor mechanical biomedical engineering prior joining usc', 'biomedical', 'activity focused', 'manufacturing process', 'engineering activity', 'digital transformation', 'dean corporate', 'national science', 'wa professor mechanical aerospace engineering north state', 'developing strong', 'engineering prior', 'main streetroom', 'joining usc', 'aerospace center', 'based maintenance', 'prior joining', 'science foundation', 'funding department', 'corporate relation', 'received funding', 'foundation various', 'wa professor', 'washington state', 'transformation dr', 'director mcnair', 'mechanical biomedical', 'material engineering', 'currently director', 'nuclear aerospace', 'published journal', 'hewlett packard', 'industry ha', 'director center predictive maintenance', 'aerospace engineering', 'mechanical system', 'biomedical engineering', 'dr bayoumi', 'mechanical behavior', 'diagnosis prognosis', 'associate dean', 'health monitoring', 'health monitoring system', 'involved developing', 'conference paper', 'nuclear aerospace engineering activity focused mechanical behavior material', 'focused mechanical', 'prognosis life', 'industry experience currently director mcnair aerospace center', 'department energy', 'behavior material', 'department defense', 'ha actively', 'mcnair aerospace', 'program mechanical', 'manager hewlett', 'professor mechanical material engineering washington state ha actively involved developing strong program mechanical', 'digital transformation dr bayoumi ha received funding department defense', 'project manager', 'packard company'}

Professor Name: Benigni, Andrea
Skill Set: {'sc', 'swearingenroom main', 'swearingenroom main streetcolumbia', 'main streetcolumbia'}

Professor Name: Berge, Nicole D.
Skill Set: {'room berge', 'stream resource', 'process development', 'production specific', 'lead carbon', 'treatment process', 'solid waste', 'landfill thermochemical', 'berge focus', 'sustainable waste', 'personal care product endocrine disrupting compound bioreactor landfill thermochemical conversion municipal solid waste increasing energy yield waste stream resource recovery waste stream leachate treatment process development subsequent evaluation innovative groundwater remediation technology', 'technique lead', 'yield waste', 'innovative groundwater', 'promote sustainable', 'value added', 'room berge focus improving understanding physical', 'energy generation', 'biological process', 'leachate treatment', 'product endocrine', 'pharmaceutical', 'value added product production specific area exploration fate disposed nanomaterials', 'waste treatment', 'development subsequent', 'energy yield', 'main street', 'waste increasing', 'understanding physical', 'treatment technique', 'added product', 'recovery waste', 'carbon sequestration', 'bioreactor landfill', 'evaluation innovative', 'focus improving', 'exploration fate', 'disrupting compound', 'endocrine disrupting', 'conversion municipal', 'product production', 'area exploration', 'manipulated promote', 'stream leachate', 'chemical', 'biological process manipulated promote sustainable waste treatment technique lead carbon sequestration', 'care product', 'municipal solid', 'specific area', 'remediation technology', 'personal care', 'improving understanding', 'fate disposed', 'groundwater remediation', 'waste stream', 'increasing energy', 'thermochemical conversion', 'subsequent evaluation', 'compound bioreactor', 'resource recovery', 'process manipulated', 'disposed nanomaterials'}

Professor Name: Besmann, Theodore  M.
Skill Set: {'nuclear fuel', 'fuel waste', 'waste material', 'nuclear fuel waste material'}

Professor Name: Bischoff, Jeff
Skill Set: {'mechanic soft', 'soft tissue', 'analysis structural constitutive evolution artificial tissue scaffolding', 'bischoff biomedical', 'passive mechanic', 'modeling growth', 'artificial tissue', 'growth tissue', 'constitutive evolution', 'dr bischoff', 'tissue scaffolding', 'hypertrophic scar', 'theoretical framework modeling growth tissue', 'framework modeling', 'passive mechanic soft tissue', 'testing modeling', 'evolution artificial', 'theoretical framework', 'modeling hypertrophic', 'dr bischoff biomedical engineering', 'biomedical engineering', 'constitutive testing', 'constitutive testing modeling hypertrophic scar', 'structural constitutive', 'analysis structural'}

Professor Name: Blanchette, James  Otto
Skill Set: {'specific project', 'developing responsive', 'islet cell', 'impact induction', 'pre angiogenic', 'cell cell', 'cell enhance', 'vascularization skiles', 'room main streetcolumbia', 'skiles l', 'orally extending', 'geometry culture', 'therapeutic tissue', 'chemotherapeutics administered', 'dimensional cell', 'instructive material', 'material tracking', 'brougham cook blanchette asc spheroid geometry culture oxygenation differentially impact induction pre angiogenic behavior endothelial cell cell transplantation', 'fall area', 'transplantation mesenchymal', 'stem cell', 'genetic engineering', 'extending proper', 'function pancreatic', 'engineering design', 'blanchette asc', 'asc spheroid', 'angiogenic behavior', 'spectrum chemotherapeutics', 'delivery therapeutic tissue engineering specific project developing responsive delivery system expand spectrum chemotherapeutics administered orally extending proper function pancreatic islet cell following transplantation genetic engineering design instructive hydrogel material tracking hypoxic signaling three dimensional cell aggregate transplantation mesenchymal stem cell enhance vascularization skiles l', 'following transplantation', 'spheroid geometry', 'aggregate transplantation', 'induction pre', 'sc fall', 'cook blanchette', 'area design', 'hanna', 'room main', 'system expand', 'proper function', 'delivery therapeutic', 'instructive hydrogel', 'transplantation genetic', 'design instructive', 'cell transplantation', 'cell instructive', 'delivery system', 'behavior endothelial', 'engineering specific', 'main streetcolumbia', 'sc fall area design cell instructive material', 'hypoxic signaling', 'mesenchymal stem', 'differentially impact', 'endothelial cell', 'project developing', 'tissue engineering', 'three dimensional', 'responsive delivery', 'signaling three', 'cell following', 'enhance vascularization', 'cell aggregate', 'tracking hypoxic', 'brougham cook', 'culture oxygenation', 'oxygenation differentially', 'administered orally', 'design cell', 'tipton', 'rucker', 'hydrogel material', 'expand spectrum', 'pancreatic islet'}

Professor Name: Boltin, Nicholas D.
Skill Set: {'predictive modeling', 'data mining', 'glean information', 'mass casualty', 'valdes', 'incident edit', 'design evaluate translational informatic tool created deploy sophisticated algorithm using latest software development human computer interaction boltin', 'application dimensional', 'network improve', 'focus developing', 'statistic', 'dimension reduction', 'article press', 'study jmir', 'focus categorized', 'initial study', 'valafar mobile decision support tool emergency department mass casualty incident edit initial study jmir mhealth uhealth', 'well predictive modeling two', 'jmir mhealth', 'including data', 'latest software', 'software development', 'supervised unsupervised', 'sophisticated algorithm', 'employ technique', 'department triage', 'mobile decision', 'science focus', 'created deploy', 'support tool', 'triage chemical', 'algorithm using', 'information data', 'unsupervised machine', 'system healthcare', 'incident article', 'using latest', 'culley jm', 'development human', 'casualty incident', 'artificial neural', 'science computer', 'broad area', 'culley', 'interdisciplinary field', 'improve emergency', 'science interdisciplinary', 'department mass', 'dr boltin', 'tool emergency', 'valafar application dimensional reduction artificial neural network improve emergency department triage chemical mass casualty incident article press boltin', 'mhealth uhealth', 'technique including', 'developing decision', 'discipline mathematics', 'reduction artificial', 'interaction boltin', 'valafar application', 'chemical mass', 'dr boltin focus developing decision support system healthcare industry data science interdisciplinary field employ technique many discipline mathematics', 'translational informatic', 'two broad', 'industry data', 'informatic tool', 'deploy sophisticated', 'healthcare industry', 'well predictive', 'decision support', 'information science computer science focus categorized two broad area one', 'design evaluate', 'technique many', 'categorized two', 'many discipline', 'area one', 'tool created', 'support system', 'press boltin', 'modeling two', 'machine learning', 'statistical analysis', 'emergency department', 'supervised unsupervised machine learning', 'computer interaction', 'field employ', 'glean information data utilizing analytical technique including data mining', 'information science', 'evaluate translational', 'neural network', 'valafar mobile', 'boltin focus', 'utilizing analytical', 'data science', 'dimensional reduction', 'edit initial', 'analytical technique', 'data utilizing', 'human computer', 'computer science'}

Professor Name: Booth, Kristen
Skill Set: {'medium frequency transformer optimization', 'frequency transformer', 'power electronics', 'power grid', 'medium frequency', 'power grid resiliency', 'transformer optimization', 'electronics reliability', 'grid resiliency', 'power electronics reliability'}

Professor Name: Brookshire, Robert G., Ph.D.
Skill Set: {'european journal operational', 'co author', 'taught new', 'performance journal itec record managementitec mainframe systemsitec linux programming administration', 'article appeared', 'north texas', 'organizational system', 'columbia', 'sc robert brookshire professor integrated information technology department college engineering computing', 'legislative study', 'european journal', 'journal past', 'system association', 'editor information', 'managementitec mainframe', 'appeared journal', 'journal past president organizational system association editor information technology', 'innovation assembly street', 'social science computer review', 'director master', 'article appeared journal computer information system', 'suite', 'master health', 'learning', 'sage publication', 'study quarterly', 'sc director master health information technology program ha taught new york', 'integrated information', 'james madison', 'journal computer', 'health information', 'author using', 'innovation assembly', 'president organizational', 'department college', 'virginia', 'texas state', 'college engineering', 'past president', 'record managementitec', 'programming administration', 'james madison co author using microcomputer sage publication', 'assembly street', 'information technology', 'professor integrated', 'technology program', 'linux programming', 'technology department', 'legislative study quarterly', 'information system', 'mainframe systemsitec', 'sc director', 'ha taught', 'brookshire professor', 'journal itec', 'association editor', 'computer review', 'social science', 'new york', 'science computer', 'performance journal', 'sc robert', 'engineering computing', 'computer information', 'north texas state', 'microcomputer sage', 'program ha', 'robert brookshire', 'madison co', 'byte', 'using microcomputer', 'journal operational', 'systemsitec linux', 'itec record'}

Professor Name: Broughton, Earnie
Skill Set: {'lead software', 'practice responsibility', 'gathering stakeholder', 'earnie lead software developer virtual test bed vtb smart ship system design project addition writing code', 'development team', 'smart ship', 'esrdc consortium', 'virtual test', 'daily activity', 'vtb smart', 'test bed', 'student programming', 'consortium education', 'project addition', 'design project', 'activity development', 'including mentoring', 'development best', 'software developer', 'industry requirement', 'writing code', 'responsibility gathering', 'professional development', 'plan integrating', 'developer virtual', 'mentoring student', 'team project', 'private industry', 'best practice', 'private industry requirement developing plan integrating requirement consortium education', 'system design', 'bed vtb', 'integrating requirement', 'requirement developing', 'programming professional', 'supervises daily', 'ship system', 'developing plan', 'including mentoring student programming professional development best practice responsibility gathering stakeholder navy', 'earnie lead', 'supervises daily activity development team project', 'addition writing', 'requirement consortium', 'stakeholder navy'}

Professor Name: Buell, Duncan A.
Skill Set: {'work ha', 'arithmetic often', 'pedagogy professor', 'oriented toward', 'ward one', 'one often need substantial parallelism', 'iphone urban', 'broad class', 'algorithm architecture', 'improvement writing', 'digital humanity', 'often multiprecision', 'evicting resident', 'project building', 'mathematics text', 'professor buell', 'co supervising', 'supervising work', 'traditional computer architecture oriented toward floating point computation normally perform significantly reduced efficiency one class problem computational problem number theory fast integer arithmetic', 'voting system', 'included algorithm', 'one neighborhood', 'machine used', 'theory fast', 'number theory', 'integer arithmetic', 'vote total', 'support vote', 'string processing', 'recent work', 'buell past', 'architecture performing', 'computer architecture', 'significantly reduced', 'ha part', 'bit arithmetic', 'directed splash', 'first year', 'floating point', 'done string', 'traditional computer', 'process lead', 'substantial parallelism', 'electronic voting', 'building custom', 'devised bit', 'machine devised', 'cc directed', 'team auditing', 'data discovering', 'whose application', 'efficiency one', 'essay linguistic', 'discrete mathematics text string processing', 'often multiprecision arithmetic', 'compute element', 'uncounted vote', 'discrete mathematics', 'splash project', 'critical interactive', 'bit bit', 'necessary another broad class problem custom computing machine devised bit oriented computation done string processing', 'vote absence', 'past included', 'text string', 'critical interactive iphone urban renewal columbia sc role evicting resident ward one neighborhood adjacent', 'reduced efficiency', 'oriented computation', 'performing computation', 'system digital', 'election data', 'ha electronic', 'point computation', 'auditing election', 'used xilinx', 'problem process', 'one often', 'part team', 'often sufficient', 'work ward', 'multiprecision arithmetic', 'one class', 'element whose', 'often need', 'sc role', 'professor buell recent work ha electronic voting system digital humanity ha part team auditing election data discovering problem process lead uncounted vote absence data support vote total certified state digital humanity', 'linguistic characteristic', 'columbia sc', 'toward floating', 'interactive iphone', 'year english', 'fast integer', 'custom computing', 'class problem', 'computation normally', 'data support', 'application programmed', 'one app', 'computational problem', 'accuracy ida', 'another broad', 'ida cc', 'absence data', 'need substantial', 'renewal columbia', 'programmed vhdl', 'discovering problem', 'resident ward', 'architecture oriented', 'characteristic improvement', 'total certified', 'urban renewal', 'problem custom', 'processing computation', 'engaged analysis first year english essay linguistic characteristic improvement writing pedagogy professor buell past included algorithm architecture performing computation', 'role evicting', 'fpgas compute', 'xilinx fpgas', 'buell recent', 'sufficient accuracy', 'neighborhood adjacent', 'co supervising work ward one app', 'computing machine', 'problem computational', 'image processing', 'writing pedagogy', 'certified state', 'computation done', 'perform significantly', 'problem number', 'lead uncounted', 'bit oriented', 'image processing computation', 'necessary another', 'bit bit arithmetic often sufficient accuracy ida cc directed splash project building custom computing machine used xilinx fpgas compute element whose application programmed vhdl', 'english essay', 'engaged analysis', 'analysis first', 'normally perform', 'humanity ha', 'state digital'}

Professor Name: Caicedo, Juan
Skill Set: {'george brown', 'dynamic intelligent', 'mechanic american', 'structural health', 'earthquake engineering structural control dr caicedo member american society civil engineer', 'structural dynamic', 'dr caicedo', 'health monitoring', 'caicedo member', 'area specialization', 'intelligent infrastructure', 'structural health monitoring', 'civil engineer', 'earthquake engineering', 'project publication', 'experimental area', 'area structural', 'brown jr', 'visit structural', 'structural control', 'structural engineering', 'engineering structural', 'caicedo area', 'control dr', 'infrastructure lab', 'model updating', 'society experimental', 'engineering educator', 'george brown jr network earthquake engineering simulation', 'list project', 'engineering institute', 'engineering simulation', 'dr caicedo area specialization structural engineering emphasis structural dynamic numerical experimental area structural dynamic', 'specialization structural', 'network earthquake', 'complete list', 'earthquake engineering institute', 'american society', 'publication visit', 'engineering emphasis', 'numerical experimental', 'dynamic numerical', 'lab website', 'jr network', 'emphasis structural', 'society engineering', 'society experimental mechanic american society engineering educator complete list project publication visit structural dynamic intelligent infrastructure lab website', 'society civil', 'educator complete', 'experimental mechanic', 'member american'}

Professor Name: Carrilho, Leo
Skill Set: {'thermal hydraulic', 'using fea cfd method', 'cfd method', 'leo carrilho', 'phenomenon structural', 'dr leo', 'pwr nuclear', 'nuclear fuel', 'using fea', 'modeling simulation', 'carrilho modeling', 'fea cfd', 'fuel core', 'structural thermal', 'process phenomenon', 'design pwr', 'dr leo carrilho modeling simulation process phenomenon structural thermal hydraulic design pwr nuclear fuel core component', 'simulation process', 'hydraulic design', 'core component'}

Professor Name: Carver, Wayne
Skill Set: {'extracellular matrix', 'seen alcohol', 'used examine', 'associated hypertension', 'non muscle', 'interconnected supported', 'endothelial cell others cell interconnected supported elaborate extracellular matrix change density organization extracellular matrix known affect cardiovascular performance correlated heart disease instance', 'hypertension myocardial', 'supported elaborate', 'heart produced', 'cardiovascular disease', 'increased accumulation', 'endothelial cell', 'stiffer myocardium', 'fibroblast behavior', 'contractile muscle', 'density organization', 'school medicine', 'particularly interested', 'room garner', 'ferry road', 'infarction increased', 'fibrosis commonly', 'fibrosis result', 'performance correlated', 'myocardial fibrosis', 'fibroblast cardiovascular', 'accumulation extracellular', 'cell others', 'animal model', 'model used', 'regulated heart', 'culture animal', 'seen fibrosis', 'excessive accumulation extracellular matrix fibrosis commonly associated hypertension myocardial infarction increased accumulation extracellular matrix seen fibrosis result stiffer myocardium', 'matrix known', 'fibrosis cell', 'chronic exposure', 'fibroblast dr', 'regulation fibroblast', 'heart particularly', 'school medicine bldg room garner ferry road columbia', 'known affect', 'focused understanding', 'largely cardiac', 'including fibroblast', 'gene expression', 'heart function', 'cell interconnected', 'excessive accumulation', 'wall heart', 'alcohol seen', 'abuse result', 'matrix change', 'alcohol abuse', 'carver lab', 'road columbia', 'understanding fibroblast', 'cardiovascular performance', 'disease instance', 'expression regulated', 'cell culture', 'cardiac fibroblast', 'exposure alcohol', 'composed contractile', 'sc ventricular', 'heart disease', 'elaborate extracellular', 'ventricular wall', 'garner ferry', 'others cell', 'bldg room', 'sc ventricular wall heart composed contractile muscle cell non muscle cell type including fibroblast', 'change density', 'result myocardial', 'examine regulation', 'commonly associated', 'alters heart', 'organization extracellular', 'behavior gene', 'matrix fibrosis', 'type including', 'affect cardiovascular', 'dr carver', 'interested chronic', 'function extracellular', 'lab focused', 'cell type', 'correlated heart', 'matrix heart', 'myocardial infarction', 'alters heart function extracellular matrix heart produced largely cardiac fibroblast dr carver lab focused understanding fibroblast behavior gene expression regulated heart particularly interested chronic exposure alcohol seen alcohol abuse result myocardial fibrosis cell culture animal model used examine regulation fibroblast cardiovascular disease', 'heart composed', 'cell non', 'produced largely', 'muscle cell', 'medicine bldg', 'result stiffer', 'matrix seen'}

Professor Name: Chandrashekhar, MVS
Skill Set: {'sic', 'chemistry', 'epitaxy', 'plasmonics', 'graphene'}

Professor Name: Chao, Yuh J.
Skill Set: {'chen bingquan', 'material structure', 'stress distortion', 'guian', 'theoretical experimental', 'welding modeling residual stress distortion', 'study failure', 'cylindrical shell', 'quasi perfect', 'experimental numerical', 'international journal', 'fuel cell', 'dukaifan', 'welding modeling', 'chao yuh', 'fatigue material', 'mechanic material', 'asme journal pressure vessel technology', 'perfect cylindrical', 'dr chao theoretical experimental study failure', 'cell system', 'biomechanics nanomechanics', 'yupeng qian', 'hao peng', 'material characterization', 'durability pem fuel cell system cao', 'yuh', 'pressure vessel', 'constraint assessment specimen tested uniaxial biaxial loading condition', 'dr chao', 'xiangtao', 'biaxial loading', 'tested uniaxial', 'journal pressure', 'combined experimental', 'failure criterion', 'vessel technology', 'characterization failure', 'yinbiao niffenegger', 'pem fuel', 'loading condition', 'markus chao', 'durability pem', 'system cao', 'wang bo', 'impact mechanic', 'bi xiangju', 'buckling quasi', 'compression combined', 'solid structure', 'fracture fatigue material structure', 'specimen tested', 'asme journal', 'journal solid', 'residual stress', 'chao theoretical', 'experimental study', 'uniaxial biaxial', 'buckling quasi perfect cylindrical shell axial compression combined experimental numerical investigation', 'international journal solid structure', 'shell axial', 'vol', 'axial compression', 'zhu shiyang', 'constraint assessment', 'numerical investigation', 'fracture fatigue', 'june wang', 'june wang bo', 'assessment specimen', 'impact mechanic material characterization failure criterion', 'modeling residual'}

Professor Name: Chaudhry, Hanif
Skill Set: {'water resource engineering', 'hydraulic transient', 'resource engineering', 'hydraulic engineering', 'water resource'}

Professor Name: Chen, Fanglin (Frank)
Skill Set: {'solid oxide electrolysis cell', 'synthesis characterization', 'solid oxide', 'electronic conducting', 'dr chen', 'gas separation', 'characterization material', 'conversion storage', 'dr chen synthesis characterization material energy conversion storage', 'catalysis electrocatalysis', 'fuel cell', 'solid oxide fuel cell', 'electrolysis cell', 'conducting ceramic', 'oxide electrolysis', 'ionic electronic', 'oxide fuel', 'separation membrane', 'chen synthesis', 'structure property', 'property relationship', 'structure property relationship', 'gas separation membrane', 'energy conversion', 'material energy', 'composition', 'ionic electronic conducting ceramic'}

Professor Name: Chen, Yinchao
Skill Set: {'sc', 'swearingenroom main', 'swearingenroom main streetcolumbia', 'main streetcolumbia'}

Professor Name: Chen, Yuche
Skill Set: {'data sensing', 'sensing analytics', 'aiaugmented simulation', 'data sensing analytics', 'emerging technology'}

Professor Name: Cheng, Xu
Skill Set: {'technology kit', 'nuclear safety', 'ha published', 'system kit', 'cheng ha', 'innovative nuclear', 'kit ha', 'field nuclear', 'director division innovative nuclear system kit ha supervised supervising phd student kit sjtu give lecture undergraduate master student ha published paper main field nuclear thermal hydraulics nuclear safety', 'sjtu china', 'experience currently', 'master student', 'phd student', 'kit germany', 'nuclear system', 'thermal hydraulics', 'supervising phd', 'institute technology', 'published paper', 'jiao tong', 'lecture undergraduate', 'hydraulics nuclear', 'ha year', 'prof cheng ha year experience year teaching experience currently professor karlsruhe institute technology kit germany shanghai jiao tong sjtu china', 'germany shanghai', 'year teaching', 'tong sjtu', 'paper main', 'director division', 'ha supervised', 'student kit', 'student ha', 'professor karlsruhe', 'give lecture', 'experience year', 'undergraduate master', 'year experience', 'teaching experience', 'karlsruhe institute', 'currently professor', 'supervised supervising', 'prof cheng', 'main field', 'nuclear thermal', 'shanghai jiao', 'division innovative', 'sjtu give', 'kit sjtu'}

Professor Name: Cole, Casey
Skill Set: {'bioinformatics', 'computational biology'}

Professor Name: Coman, Paul
Skill Set: {'thermodynamics', 'liion safety'}

Professor Name: Crichigno, Jorge
Skill Set: {'programmable switch', 'science dmz', 'iot security'}

Professor Name: Cui, Taixing
Skill Set: {'fourth military', 'military medical', 'garner ferry', 'school medicine', 'fourth military medical', 'sc', 'molecular medicine', 'bldg room', 'japan', 'room garner', 'ferry road', 'medicine bldg', 'ehime school medicine', 'road columbia', 'r china', 'school medicine bldg room garner ferry road columbia', 'ehime school'}

Professor Name: De Backer, Wout
Skill Set: {'fused deposition modeling', 'printing', 'additive manufacturing', 'deposition modeling', 'fused deposition'}

Professor Name: Deng, Xiaomin
Skill Set: {'process simulation', 'molecular dynamic simulation', 'finite element', 'modeling simulation', 'manufacturing process simulation', 'molecular dynamic', 'manufacturing process', 'dynamic simulation', 'solid mechanic'}

Professor Name: Devereux, Emily
Skill Set: {'based principal', 'dependency theory', 'past year', 'ranking perceived', 'foundation agency', 'agent resource', 'resource dependency', 'bias funding', 'community economic', 'capacity reputation', 'administrator year', 'reputation goal', 'year administration', 'based administrative', 'review process', 'funding sponsor', 'past year administration leadership experience ha concentrated strategic growth', 'senior administrator year administration community economic development', 'concentrated strategic', 'strategic planning', 'signal competitiveness', 'strategic growth', 'leadership experience', 'experience ha', 'funding review', 'perceived bias', 'administration leadership', 'focused national', 'national ranking', 'process based', 'administrative capacity', 'administration focused', 'federal foundation', 'administration focused national ranking perceived bias funding review process based administrative capacity reputation goal identify contributes perceived bias federal foundation agency review enables institution signal competitiveness funding sponsor based principal agent resource dependency theory', 'agency review', 'competitiveness funding', 'ha concentrated', 'review enables', 'sponsor based', 'identify contributes', 'senior administrator', 'development', 'institution signal', 'administration community', 'economic development', 'principal agent', 'student growth', 'bias federal', 'contributes perceived', 'goal identify', 'enables institution'}

Professor Name: Dillon, Anthony L
Skill Set: {'technology clinical', 'partnership relationship', 'related educational', 'undergraduate graduate', 'brings numerous', 'dillon director', 'within iit', 'sector related', 'innovation assembly street', 'suite', 'director internship', 'department integrated', 'integrated information', 'business partnership', 'innovation assembly', 'educational technology', 'numerous business', 'private public', 'assembly street', 'information technology', 'worked within', 'program within', 'technology brings', 'graduate program', 'mhit program', 'within private', 'instructor undergraduate', 'sc dillon director internship department integrated information technology clinical instructor undergraduate graduate program within iit mhit program ha worked within private public sector related educational technology brings numerous business partnership relationship', 'sc dillon', 'public sector', 'internship department', 'clinical instructor', 'ha worked', 'iit mhit', 'program ha'}

Professor Name: Dougal, Roger A.
Skill Set: {'hybrid power', 'sponsorship office', 'better realize', 'prof dougal', 'bed software', 'energy system', 'dougal ha', 'several post doctoral scholar faculty', 'electrochemical power', 'related new', 'main street', 'capacity oversees', 'range associated', 'electric system', 'comprehensive simulation virtual prototyping environment multidisciplinary dynamic system environment applied study electric system navy ship', 'grape joint', 'encompasses wide', 'sc dougal lead power energy system group', 'sponsored industry', 'electronics also', 'engineering department', 'electronics utility', 'grid since', 'dougal currently', 'dozen graduate', 'currently supervises', 'graduate student', 'associated technology', 'environment applied', 'site director', 'electrochemical power source', 'number engineering', 'power generation', 'advanced power', 'department dr', 'navy ship', 'dynamic system', 'power electronic', 'control dr dougal currently supervises dozen graduate student', 'sponsorship office naval', 'dr dougal', 'dougal member', 'member school', 'arkansas seek', 'grid better', 'electric ship', 'nsf sponsored', 'street columbia', 'school prof', 'several post', 'center grid', 'wide range', 'multidisciplinary dynamic', 'technology across', 'lead power', 'simulation virtual', 'coordination activity member school prof dougal also site director new nsf sponsored industry cooperative center grid connected advanced power electronic system grape joint project usc arkansas seek insert greater level advanced power electronics utility power grid better realize smart grid since', 'member board', 'consortium esrdc', 'post doctoral', 'coordination activity', 'industry cooperative', 'seek insert', 'oversees usc', 'power energy', 'insert greater', 'system environment', 'joint project', 'number undergraduate researcher', 'power grid', 'project usc', 'also encompasses', 'usc activity', 'new power', 'test bed', 'supervises dozen', 'connected advanced', 'board director', 'smart grid', 'also site', 'director new', 'prof dougal ha overseen development virtual test bed software', 'undergraduate researcher', 'system group', 'principally focus', 'electronic system', 'control dr', 'usc arkansas', 'virtual prototyping', 'capacity oversees usc activity related new power generation', 'greater level', 'level advanced', 'power source', 'distribution technology', 'applied study', 'environment multidisciplinary', 'power electronics', 'technology ship', 'across number', 'swearingenroom main', 'new nsf', 'scholar faculty', 'principally focus power electronics also encompasses wide range associated technology across number engineering department dr dougal member board director electric ship development consortium esrdc', 'activity related', 'activity member', 'prototyping environment', 'focus power', 'processing', 'virtual test', 'development virtual', 'doctoral scholar', 'ha overseen', 'director electric', 'system navy', 'realize smart', 'development consortium', 'sc dougal', 'distribution technology ship', 'swearingenroom main street columbia', 'hybrid power source', 'comprehensive simulation', 'number undergraduate', 'office naval', 'cooperative center', 'overseen development', 'system grape', 'utility power', 'study electric', 'grid connected', 'dougal lead', 'ship development', 'dougal also'}

Professor Name: Downey, Austin
Skill Set: {'health monitoring', 'realtime system', 'structural health', 'structural health monitoring'}

Professor Name: Dryer, Frederick  L.
Skill Set: {'chemical kinetic', 'chemical kinetics relevant space air breathing propulsion', 'emission interaction', 'gas phase', 'alternative fuel', 'incineration', 'conversion related', 'space air', 'emission generation', 'transportation stationary', 'application driven', 'chemical kinetic property', 'property non', 'relevant space', 'combustion property', 'main street', 'derived fuel', 'combustion related pollutant mitigation', 'ha year', 'phase interaction', 'effect stationary', 'gravity environment', 'pollutant concern', 'related issue', 'carbon cycle', 'material emission', 'concern fire', 'interaction related', 'physical chemistry', 'sc dryer ha year application driven fundamental experience thermal science heat transfer', 'emission internal', 'power generation', 'including gasoline', 'reduction net', 'energy security', 'interaction including', 'emission generation abatement petroleum derived fuel', 'internal combustion', 'aerosol particulate', 'phase gas', 'safety related', 'kinetics fuel', 'dryer current', 'abatement petroleum', 'cycle emission', 'particle burning', 'sc dryer', 'hazardous waste', 'including chlorine', 'catalyst material', 'environment solid', 'heavy fuel oil combustion property non petroleum derived alternative fuel', 'energy conversion', 'dryer ha', 'fuel hazardous', 'kinetics relevant', 'petroleum derived', 'aerosol particulate emission interaction including chlorine', 'driven fundamental', 'burning phenomenon', 'heat transfer', 'air breathing', 'fire safety', 'security reduction', 'component effect', 'chemical kinetics', 'ability address energy security reduction net carbon cycle emission well pollutant concern fire safety related issue earth micro gravity environment solid phase gas phase interaction related particle burning phenomenon nano catalyst material emission internal combustion engine', 'nano catalyst', 'stationary power', 'science heat', 'thermal science', 'nitrogen oxide', 'micro gravity', 'production', 'year application', 'fundamental experience', 'generation abatement', 'combustion', 'stationary energy', 'sulfur ash metal component effect stationary energy conversion', 'sulfur ash', 'issue earth', 'including hydrocarbon', 'fluid dynamic', 'chemical processing', 'ability address', 'dr dryer', 'chemistry chemical', 'fuel oil', 'non petroleum', 'derived alternative', 'solid phase', 'particulate emission', 'diesel', 'kinetic property', 'related ignition', 'combustion related', 'net carbon', 'gas turbine', 'breathing propulsion', 'combustion engine', 'ash metal', 'related particle', 'fire safety dr dryer current chemistry chemical kinetics fuel hazardous waste material related ignition', 'oil combustion', 'experience thermal', 'emission well', 'fossil renewable', 'renewable energy', 'metal component', 'related pollutant', 'safety dr', 'related ground', 'waste material', 'earth micro', 'rm carolinacolumbia', 'well pollutant', 'pollutant mitigation', 'ground transportation', 'heavy fuel', 'current chemistry', 'material related', 'fossil renewable energy conversion related ground transportation stationary power generation', 'phenomenon nano', 'address energy'}

Professor Name: Eastman, Caroline M.
Skill Set: {'search interface', 'professor eastman focus effective efficient algorithm information retrieval want information', 'want information', 'correct find', 'database security', 'efficient algorithm', 'easily quickly', 'eastman focus', 'file organization', 'professor eastman', 'focus effective', 'efficient file organization search interface user interaction ha also working colleague problem database security pi co director professor bowles nsf experience undergraduate site', 'interaction ha', 'ha also', 'director professor', 'easily quickly find useful correct find work area ha addressed broad spectrum problem', 'user interaction', 'bowles nsf', 'useful correct', 'work area', 'area ha', 'colleague problem', 'nsf experience', 'experience undergraduate', 'organization search', 'retrieval want', 'find useful', 'interface user', 'effective efficient', 'find work', 'pi co', 'ha addressed', 'security pi', 'spectrum problem', 'efficient file', 'undergraduate site', 'professor bowles', 'information retrieval', 'addressed broad', 'problem database', 'working colleague', 'algorithm information', 'broad spectrum', 'quickly find', 'co director', 'also working'}

Professor Name: Eberth, John F.
Skill Set: {'biomechanics', 'biomedical engineering', 'mechanobiology', 'cardiovascular'}

Professor Name: Ebner, Armin D.
Skill Set: {'found medical', 'fine ferromagnetic', 'emphasis use', 'today major', 'consists use', 'catholic chile', 'ebner major', 'gradient magnetic', 'assist magnetic', 'field separation', 'magnetic drug', 'ebner major application magnetic field separation', 'size low', 'fluid medium', 'separation hgms', 'field become', 'used assist', 'major emphasis', 'particular emphasis use high gradient magnetic separation hgms principle technique consists use fine ferromagnetic element result application external magnetic field become energized create magnetic gradient large enough collect particle small size low magnetism collection otherwise realized today major emphasis potential hgms found medical area technique used assist magnetic drug targeting cancer treatment', 'hgms found', 'particle small', 'use fine', 'collect particle', 'medical area', 'application external', 'result application', 'magnetic particle', 'drug targeting', 'collection manipulation magnetic particle fluid medium', 'collection otherwise', 'technique used', 'embolization control', 'external magnetic', 'technique consists', 'control well', 'magnetic field', 'high gradient', 'principle technique', 'particle fluid', 'embolization control well rapid detoxification', 'chile', 'particular emphasis', 'magnetic separation', 'cancer treatment', 'small size', 'hgms principle', 'potential hgms', 'otherwise realized', 'manipulation magnetic', 'area technique', 'collection manipulation', 'ph', 'restenosis', 'enough collect', 'gradient large', 'magnetism collection', 'energized create', 'element result', 'application magnetic', 'use high', 'ferromagnetic element', 'low magnetism', 'emphasis potential', 'well rapid', 'major application', 'become energized', 'large enough', 'realized today', 'targeting cancer', 'rapid detoxification', 'magnetic gradient', 'create magnetic'}

Professor Name: Eslambolchi Moghadam, Sara
Skill Set: {'wound healing', 'medicinal plant', 'natural product', 'drug discovery'}

Professor Name: Fan, Daping
Skill Set: {'including hypoxia', 'developing strategy', 'response form', 'deposit disease', 'network developing', 'killer united', 'dendritic cell', 'killer united state many developed country atherosclerosis lipid deposit disease chronic inflammatory condition macrophage primary cholesterol sink well dominant inflammation machine atherogenesis macrophage foam cell formation inflammatory response form vicious cycle promote atherogenesis goal laboratory develop strategy disrupt vicious cycle promote regression atherosclerotic plaque restoring macrophage cholesterol homeostasis controlling macrophage inflammation myeloid cell major cell component tumor microenvironment tme', 'tme mainly', 'improve cancer', 'feature tme', 'restoring macrophage', 'cell formation', 'toll like', 'mainly myeloid', 'unusual metabolic', 'cancer treatment', 'cell component', 'understanding signal', 'machine atherogenesis', 'malignant cell', 'network tune', 'due unique feature tme including hypoxia', 'controlling macrophage', 'sc rupture', 'resulting thrombosis', 'atherogenesis goal', 'formation inflammatory', 'manipulate improve', 'many developed', 'metabolic profile', 'school medicine', 'macrophage cholesterol', 'room garner', 'ferry road', 'derived suppressor', 'like receptor', 'tumor associated', 'thrombosis direct', 'signaling network', 'goal laboratory', 'function myeloid', 'tumor associated macrophage tam advanced tumor', 'whose interaction', 'sc rupture vulnerable atherosclerotic plaque resulting thrombosis direct cause heart attack stroke', 'major cell', 'tumor myeloid', 'form vicious', 'favor cancer', 'sink well', 'suppressor cell', 'tumor microenvironment', 'cholesterol sink', 'atherosclerotic plaque', 'school medicine bldg room garner ferry road columbia', 'united state', 'macrophage foam', 'atherosclerosis lipid', 'focused understanding', 'determines aggressiveness', 'cytokine chemokines', 'cholesterol homeostasis', 'aggressiveness tumor', 'macrophage inflammation', 'promote regression', 'state many', 'cell tme', 'component tumor', 'myeloid derived', 'associated macrophage', 'tam advanced', 'cancer progression', 'plaque resulting', 'complex signaling network tune dynamic function myeloid cell favor cancer progression focused understanding signal network developing strategy manipulate improve cancer treatment', 'attack stroke', 'vicious cycle', 'cell favor', 'inflammation myeloid', 'strategy disrupt', 'inflammatory response', 'microenvironment tme', 'cell major', 'abundance cytokine chemokines toll like receptor tlr ligand', 'road columbia', 'lipid deposit', 'dynamic function', 'regression atherosclerotic', 'heart attack', 'strategy manipulate', 'promote atherogenesis', 'laboratory develop', 'direct cause', 'homeostasis controlling', 'country atherosclerosis', 'garner ferry', 'foam cell', 'bldg room', 'atherogenesis macrophage', 'interaction malignant', 'cause heart', 'receptor tlr', 'macrophage primary', 'vulnerable atherosclerotic', 'plaque restoring', 'inflammatory condition', 'develop strategy', 'developed country', 'unique feature', 'disease chronic', 'complex signaling', 'tune dynamic', 'signal network', 'advanced tumor', 'macrophage tam', 'whose interaction malignant cell determines aggressiveness tumor myeloid cell tme mainly myeloid derived suppressor cell mdscs', 'chemokines toll', 'cell mdscs', 'disrupt vicious', 'cycle promote', 'cell determines', 'tlr ligand', 'dominant inflammation', 'primary cholesterol', 'rupture vulnerable', 'well dominant', 'inflammation machine', 'myeloid cell', 'condition macrophage', 'chronic inflammatory', 'abundance cytokine', 'due unique', 'medicine bldg', 'tme including', 'unusual metabolic profile', 'progression focused'}

Professor Name: Farkas, Csilla
Skill Set: {'information security'}

Available Proposals:

Title: Advanced Technological Education
Skills Required: {'industry', 'higher education', 'engineering', 'professional development', 'development', 'science engineering', 'material', 'economic development'}

Title: Environmental Convergence Opportunities in Chemical, Bioengineering, Environmental, and Transport Systems
Skills Required: {'engineering', 'science engineering', 'environmental engineering', 'chemical'}

Title: Collaborative Research in Computational Neuroscience
Skills Required: {'theoretical foundation', 'computer science'}

Title: National Artificial Intelligence (AI) Research Institutes
Skills Required: {'industry', 'continued advancement', 'economic impact', 'development', 'security', 'machine learning', 'learning', 'ai', 'artificial intelligence'}

Title: Computer and Information Science and Engineering (CISE): Core Programs
Skills Required: {'science engineering', 'project develop', 'engineering', 'information science', 'communication', 'computer information'}

Title: Division of Materials Research: Topical Materials Research Programs: Biomaterials (BMAT), Condensed Matter Physics (CMP), Metals and Metallic Nanostructures (MMN), Polymers (POL) (DMR-TMRP BMAT, CMP, MMN, POL) (nsf20589) | NSF - National Science Foundation
Skills Required: {'property', 'industry', 'design synthesis', 'polymer', 'physic', 'engineering', 'biomaterials', 'novel design', 'national science', 'united state', 'development', 'material unique', 'control', 'science foundation', 'fundamental understanding', 'lead new', 'synthesis characterization', 'material', 'processing', 'synthesis', 'discovery'}

Title: Division of Materials Research: Topical Materials Research Programs: Ceramics (CER), Electronic and Photonic Materials (EPM), Solid State and Materials Chemistry (SSMC)
Skills Required: {'property', 'industry', 'design synthesis', 'engineering', 'novel design', 'united state', 'development', 'material unique', 'control', 'chemistry', 'fundamental understanding', 'lead new', 'synthesis characterization', 'material', 'processing', 'synthesis', 'solid state', 'discovery', 'electronic photonic'}

Title: Re-entry to Active Research Program
Skills Required: {'chemistry', 'chemical'}

Title: Research Experiences for Teachers (RET) in Engineering and Computer Science (nsf20584) | NSF - National Science Foundation
Skills Required: {'industry', 'revolve around', 'engineering', 'national science', 'computer information', 'directorate engineering', 'science technology', 'science foundation', 'science engineering', 'experience new', 'material', 'area', 'information science', 'computer science'}

Title: Condensed Matter and Materials Theory (CMMT) (nsf20582) | NSF - National Science Foundation
Skills Required: {'national science', 'science foundation', 'electronic photonic', 'broad spectrum', 'polymer', 'molecular dynamic', 'material soft', 'material state', 'soft material', 'material', 'statistical mechanic', 'physic', 'biomaterials', 'polymeric material', 'dynamic', 'area', 'learning', 'material system', 'solid state', 'property', 'development', 'chemistry', 'fundamental understanding', 'soft matter', 'method used', 'machine learning', 'modeling'}

Title: Division of Chemistry: Disciplinary Research Programs
Skills Required: {'imaging', 'computational method', 'catalysis', 'chemistry', 'dynamic', 'synthesis', 'chemical'}

Title: Joint DMS/NIGMS Initiative to Support Research at the Interface of the Biological and Mathematical Sciences
Skills Required: {'support interface', 'statistic', 'biomedical', 'health', 'national science', 'science foundation', 'biomedical science', 'national institute'}

Title: Centers for Chemical Innovation
Skills Required: {'chemical', 'communication'}

Title: NSF Program on Fairness in Artificial Intelligence in Collaboration with Amazon
Skills Required: {'ha long', 'algorithm', 'health', 'security', 'ai system', 'decision support', 'support system', 'machine learning', 'learning', 'ai', 'artificial intelligence'}

Title: Opportunities for Promoting Understanding through Synthesis
Skills Required: {'synthesis'}

Title: Cyber-Physical Systems
Skills Required: {'personalized medicine', 'automation', 'civil', 'energy', 'security', 'machine learning', 'learning', 'range application', 'artificial intelligence'}

Title: Hydrologic Sciences (HS)
Skills Required: {'biological process', 'engineering', 'experimental theoretical', 'energy', 'science engineering', 'modeling', 'synthesis', 'including limited', 'energy transport', 'chemical'}

Title: Spectrum Innovation Initiative: National Center for Wireless Spectrum Research
Skills Required: {'system application', 'communication', 'thing iot', 'high speed', 'application including', 'wireless communication', 'internet thing'}

Title: Gen-4 Engineering Research Centers
Skills Required: {'including engineering', 'engineering', 'development'}

Title: Future Manufacturing
Skills Required: {'logistics', 'machine design', 'production', 'business', 'material', 'chemical'}

Title: Expeditions in Computing (Expeditions) (nsf20544) | NSF - National Science Foundation
Skills Required: {'science foundation', 'science engineering', 'engineering', 'information science', 'national science', 'computer information'}

Title: Reproducible Cells and Organoids via Directed-Differentiation Encoding
Skills Required: {'cell cell', 'foundation nsf', 'national science', 'control', 'science foundation', 'fundamental understanding', 'product', 'develop strategy', 'dynamic', 'cell type', 'chemical'}

Title: NSF-Simons Research Collaborations on the Mathematical and Scientific Foundations of Deep Learning
Skills Required: {'deep learning', 'graduate student', 'principal investigator', 'engineering', 'activity focused', 'national science', 'computer information', 'science foundation', 'science engineering', 'area', 'learning', 'information science', 'artificial intelligence', 'new york'}

Title: Computer Science for All
Skills Required: {'foundation nsf', 'professional development', 'national science', 'development', 'science foundation', 'material', 'teaching', 'computer science'}

Title: National Robotics Initiative 2.0: Ubiquitous Collaborative Robots
Skills Required: {'united state', 'development', 'robotics'}

Title: Environmental Convergence Opportunities in Chemical, Bioengineering, Environmental, and Transport Systems
Skills Required: {'science engineering', 'engineering', 'environmental engineering', 'chemical'}

Title: Navigating the New Arctic (NNA) (nsf20514) | NSF - National Science Foundation
Skills Required: {'foundation nsf', 'engineering', 'program director', 'national science', 'science foundation', 'science engineering', 'area future', 'area'}

Title: Understanding the Rules of Life: Microbiome Theory and Mechanisms (URoL:MTM) (nsf20513) | NSF - National Science Foundation
Skills Required: {'foundation nsf', 'engineering', 'program director', 'national science', 'science foundation', 'science engineering', 'area future', 'area'}

Title: Addressing Systems Challenges through Engineering Teams (ASCENT) (nsf20511) | NSF - National Science Foundation
Skills Required: {'system application', 'big data', 'communication', 'engineering', 'national science', 'energy', 'control', 'science foundation', 'security', 'cyber system', 'machine learning', 'synthesis', 'area', 'learning', 'fuel', 'information science', 'artificial intelligence'}

Title: NSF Engineering - UKRI Engineering and Physical Sciences Research Council Lead Agency Opportunity
Skills Required: {'division electrical', 'communication', 'review process', 'engineering', 'civil', 'national science', 'directorate engineering', 'science foundation', 'cyber system', 'chemical'}

Title: Transitions to Excellence in Molecular and Cellular Biosciences Research (Transitions) (nsf20505) | NSF - National Science Foundation
Skills Required: {'retention', 'professional development', 'track', 'national science', 'development', 'science foundation', 'full professor', 'area', 'year'}

Title: National Artificial Intelligence (AI) Research Institutes
Skills Required: {'industry', 'continued advancement', 'economic impact', 'development', 'security', 'machine learning', 'learning', 'ai', 'artificial intelligence'}

Title: NSF/CASIS Collaboration on Transport Phenomena Research on the International Space Station (ISS) to Benefit Life on Earth
Skills Required: {'thermal transport', 'transport process', 'foundation nsf', 'engineering', 'national science', 'science foundation', 'combustion', 'transport phenomenon', 'fluid dynamic', 'dynamic', 'chemical'}

Title: NSF/CASIS Collaboration on Tissue Engineering and Mechanobiology on the International Space Station (ISS) to Benefit Life on Earth (nsf20500) | NSF - National Science Foundation
Skills Required: {'foundation nsf', 'engineering', 'civil', 'national science', 'mechanobiology', 'science foundation', 'tissue engineering', 'chemical'}

Title: IUSE / Professional Formation of Engineers:
Skills Required: {'engineering material', 'engineering education', 'engineering department', 'capstone design', 'first year', 'engineering', 'professional engineer', 'material', 'learning', 'professional practice', 'year'}

Title: Formal Methods in the Field
Skills Required: {'system application', 'embedded system', 'engineering', 'computer information', 'science engineering', 'machine learning', 'modeling', 'synthesis', 'area', 'learning', 'information science', 'year'}

Title: EMERGING FRONTIERS IN RESEARCH AND INNOVATION
Skills Required: {'area', 'engineering', 'directorate engineering'}

Title: NSF/DOE Partnership in Basic Plasma Science and Engineering
Skills Required: {'property', 'physic', 'engineering', 'material science', 'applied mathematics', 'science engineering', 'material', 'plasma'}

Title: Computer and Information Science and Engineering (CISE): Core Programs
Skills Required: {'science engineering', 'project develop', 'engineering', 'information science', 'communication', 'computer information'}

Title: Division of
Skills Required: {'imaging', 'computational method', 'catalysis', 'chemistry', 'dynamic', 'synthesis', 'chemical'}

Title: Methodology, Measurement, and Statistics
Skills Required: {'mm', 'statistic', 'development'}

Title: Secure and Trustworthy Cyberspace Frontiers
Skills Required: {'engineering', 'security', 'cyber system', 'problem involving'}

Title: NSF Program on Fairness in Artificial Intelligence in Collaboration with Amazon (FAI) (nsf19571) | NSF - National Science Foundation
Skills Required: {'ha long', 'health', 'national science', 'science foundation', 'security', 'ai system', 'machine learning', 'learning', 'ai', 'artificial intelligence'}

Title: Cyber-Physical Systems
Skills Required: {'personalized medicine', 'civil', 'energy', 'security', 'range application', 'artificial intelligence'}

Title: National Robotics Initiative 2.0: Ubiquitous Collaborative Robots (NRI-2.0) (nsf19536) | NSF - National Science Foundation
Skills Required: {'national science', 'united state', 'development', 'science foundation', 'robotics'}

Title: Enabling Quantum Leap: Quantum Idea Incubator for Transformational Advances in Quantum Systems (QII - TAQS) (nsf19532) | NSF - National Science Foundation
Skills Required: {'foundation nsf', 'engineering', 'program director', 'national science', 'science foundation', 'science engineering', 'area future', 'area'}

Title: Spectrum Efficiency, Energy Efficiency, and Security (SpecEES):
Skills Required: {'engineering', 'national science', 'directorate engineering', 'computer information', 'energy', 'science foundation', 'science engineering', 'service', 'security', 'information science'}

Title: CNH2: Dynamics of Integrated Socio-Environmental Systems
Skills Required: {'dynamic', 'chemical'}

Title: NSF/CASIS Collaboration on Transport Phenomena Research on the International Space Station (ISS) to Benefit Life on Earth
Skills Required: {'thermal transport', 'transport process', 'foundation nsf', 'engineering', 'national science', 'science foundation', 'combustion', 'transport phenomenon', 'fluid dynamic', 'dynamic', 'chemical'}

Title: Methodology, Measurement, and Statistics (MMS) (nsf19520) | NSF - National Science Foundation
Skills Required: {'statistic', 'national science', 'development', 'science foundation', 'mm', 'production'}

Title: Designing Materials to Revolutionize and Engineer our Future (DMREF) (nsf19516) | NSF - National Science Foundation
Skills Required: {'property', 'system design', 'engineering', 'material science', 'national science', 'development', 'coordination activity', 'science foundation', 'science engineering', 'material', 'advanced material', 'discovery'}

Title: Navigating the New Arctic
Skills Required: {'foundation nsf', 'engineering', 'program director', 'national science', 'science foundation', 'science engineering', 'area future', 'area'}

Title: NSF/CASIS Collaboration on Tissue Engineering and Mechanobiology on the International Space Station (ISS) to Benefit Life on Earth (nsf19509) | NSF - National Science Foundation
Skills Required: {'foundation nsf', 'engineering', 'civil', 'national science', 'mechanobiology', 'science foundation', 'tissue engineering', 'chemical'}

Title: Scalable Parallelism in the Extreme (SPX) (nsf19505) | NSF - National Science Foundation
Skills Required: {'national science', 'science foundation', 'year'}

Title: Big Data Regional Innovation Hubs
Skills Required: {'industry', 'big data', 'engineering', 'united state', 'computer information', 'science engineering', 'information science'}

Title: Formal Methods in the Field
Skills Required: {'system application', 'engineering', 'computer information', 'science engineering', 'machine learning', 'modeling', 'synthesis', 'area', 'learning', 'information science', 'year'}

Title: Infrastructure Innovation for Biological Research
Skills Required: {'multidisciplinary approach'}

Title: Collaborative Research in Computational Neuroscience
Skills Required: {'theoretical foundation', 'computer science'}

Title: Division of Molecular and Cellular Biosciences: Investigator-initiated research projects (MCB) (nsf18585) | NSF - National Science Foundation
Skills Required: {'national science', 'science foundation'}

Title: Secure and Trustworthy Cyberspace
Skills Required: {'engineering', 'security', 'cyber system', 'problem involving'}

Title: Advanced Technological Education
Skills Required: {'industry', 'higher education', 'engineering', 'professional development', 'development', 'science engineering', 'material'}

Title: Information and Intelligent Systems (IIS): Core Programs
Skills Required: {'project develop', 'intelligent system'}

Title: Computer and Network Systems (CNS): Core Programs
Skills Required: {'development novel', 'development'}

Title: Office of Advanced Cyberinfrastructure (OAC): Research Core Program
Skills Required: {'engineering education', 'engineering', 'development', 'science engineering', 'service', 'security', 'area', 'translational', 'discovery', 'computational data'}

Title: Joint DMS/NIGMS Initiative to Support Research at the Interface of the Biological and Mathematical Sciences
Skills Required: {'support interface', 'statistic', 'foundation nsf', 'biomedical', 'health', 'national science', 'science foundation', 'biomedical science', 'national institute', 'nih'}

Title: Division of Physics: Investigator-Initiated Research Projects
Skills Required: {'broad range', 'physic'}

Title: Division of
Skills Required: {'imaging', 'computational method', 'catalysis', 'chemistry', 'dynamic', 'synthesis', 'chemical'}

Title: Smart and Autonomous Systems
Skills Required: {'perception', 'communication', 'actuation', 'smart grid'}

Title: NSF/FDA SCHOLAR-IN-RESIDENCE AT FDA (nsf18556) | NSF - National Science Foundation
Skills Required: {'system directorate', 'graduate student', 'principal investigator', 'foundation nsf', 'engineering', 'health', 'national science', 'computer information', 'directorate engineering', 'science foundation', 'science engineering', 'engineering division', 'material', 'information science', 'computer science'}

Title: Smart and Connected Health
Skills Required: {'public health', 'process modeling', 'foundation nsf', 'biomedical', 'engineering', 'health', 'national science', 'development', 'computer information', 'science foundation', 'science engineering', 'national institute', 'security', 'modeling', 'area', 'aging', 'nih', 'information science'}

Title: Cyber-Physical Systems (CPS) (nsf18538) | NSF - National Science Foundation
Skills Required: {'personalized medicine', 'civil', 'national science', 'energy', 'science foundation', 'security', 'range application', 'artificial intelligence'}

Title: Computer Science for All
Skills Required: {'foundation nsf', 'professional development', 'national science', 'development', 'science foundation', 'material', 'teaching', 'computer science'}

Title: Formal Methods in the Field
Skills Required: {'system application', 'engineering', 'computer information', 'science engineering', 'machine learning', 'modeling', 'synthesis', 'area', 'learning', 'information science', 'year'}

Title: Expeditions in Computing
Skills Required: {'science engineering', 'engineering', 'information science', 'computer information'}

Title: Re-entry to Active Research Program
Skills Required: {'chemical'}

Title: NSF/CASIS Collaboration on Fluid Dynamics and Particulate and Multiphase Processes Research on the International Space Station to Benefit Life on Earth
Skills Required: {'foundation nsf', 'engineering', 'national science', 'science foundation', 'fluid dynamic', 'dynamic', 'chemical'}

Title: National Robotics Initiative 2.0: Ubiquitous Collaborative Robots (NRI-2.0) (nsf18518) | NSF - National Science Foundation
Skills Required: {'national science', 'united state', 'development', 'science foundation', 'robotics'}

Title: Resource Implementations for Data Intensive Research in the Social, Behavioral and Economic Sciences
Skills Required: {'product', 'area'}

Title: NSF/CASIS Collaboration on Tissue Engineering on the International Space Station to Benefit Life on Earth
Skills Required: {'foundation nsf', 'engineering', 'national science', 'science foundation', 'tissue engineering', 'chemical'}

Title: EPSCoR Research Infrastructure Improvement Program: Track-2 Focused EPSCoR Collaborations (RII Track-2 FEC)
Skills Required: {'industry', 'higher education', 'foundation nsf', 'national science', 'development', 'science foundation', 'program level'}

Title: Collaborative Research in Computational Neuroscience
Skills Required: {'theoretical foundation', 'computer science'}

Title: Condensed Matter and Materials Theory
Skills Required: {'electronic photonic', 'broad spectrum', 'polymer', 'molecular dynamic', 'material soft', 'material state', 'soft material', 'material', 'statistical mechanic', 'physic', 'biomaterials', 'polymeric material', 'dynamic', 'area', 'learning', 'material system', 'solid state', 'property', 'development', 'chemistry', 'fundamental understanding', 'soft matter', 'method used', 'machine learning', 'modeling'}

Title: Spectrum Efficiency, Energy Efficiency, and Security (SpecEES): Enabling Spectrum for All
Skills Required: {'engineering', 'national science', 'directorate engineering', 'computer information', 'energy', 'science foundation', 'science engineering', 'service', 'security', 'information science'}

Title: Scalable Parallelism in the Extreme
Skills Required: {'year'}

Title: Division of Molecular and Cellular Biosciences: Investigator-Initiated Research Projects
Skills Required: {'general'}

Title: Division of Materials Research: Topical Materials Research Programs
Skills Required: {'property', 'industry', 'design synthesis', 'engineering', 'novel design', 'united state', 'development', 'material unique', 'control', 'fundamental understanding', 'lead new', 'synthesis characterization', 'material', 'processing', 'synthesis', 'discovery'}

Title: Emerging Frontiers in Research and Innovation 2018
Skills Required: {'area', 'engineering', 'directorate engineering'}

Title: Secure and Trustworthy Cyberspace
Skills Required: {'engineering', 'security', 'cyber system', 'problem involving'}

Title: Research Experiences for Teachers (RET) in Engineering and Computer Science
Skills Required: {'revolve around', 'graduate student', 'engineering', 'computer information', 'directorate engineering', 'science technology', 'science engineering', 'experience new', 'area', 'information science', 'computer science'}

Title: Information and Intelligent Systems (IIS): Core Programs
Skills Required: {'project develop', 'intelligent system'}

Title: Joint DMS/NIGMS Initiative to Support Research at the Interface of the Biological and Mathematical Sciences
Skills Required: {'support interface', 'statistic', 'foundation nsf', 'biomedical', 'health', 'national science', 'science foundation', 'biomedical science', 'national institute', 'nih'}

Title: Advanced Technological Education (ATE) (nsf17568) | NSF - National Science Foundation
Skills Required: {'industry', 'engineering', 'professional development', 'national science', 'development', 'science foundation', 'science engineering', 'material'}

Title: Geophysics
Skills Required: {'property', 'physic', 'wave propagation', 'theoretical experimental', 'experimental study', 'composition', 'material'}

Title: Mathematical Sciences Research Institutes (nsf17553) | NSF - National Science Foundation
Skills Required: {'statistic', 'national science', 'united state', 'science foundation', 'discovery'}

Title: Ideas Lab: Practical Fully-Connected Quantum Computer Challenge (PFCQC)
Skills Required: {'processing', 'information processing', 'quantum computing', 'physic'}

Title: Robert Noyce Teacher Scholarship Program (nsf17541) | NSF - National Science Foundation
Skills Required: {'including engineering', 'engineering', 'track', 'national science', 'science technology', 'science foundation', 'teaching', 'computer science'}

Title: Building Community and Capacity in Data Intensive Research in Education
Skills Required: {'engineering', 'engineering activity', 'creating new', 'science engineering', 'area'}

Title: Innovations at the Nexus of Food, Energy and Water Systems
Skills Required: {'energy'}

Title: Computer Science for All
Skills Required: {'foundation nsf', 'professional development', 'national science', 'development', 'science foundation', 'material', 'teaching', 'computer science'}

Title: Integrative Strategies for Understanding Neural and Cognitive Systems
Skills Required: {'engineering', 'science engineering', 'multidisciplinary approach', 'area', 'broad spectrum'}
'''


In [ ]:
content = '''
Proposal Name;Professor Name
Advanced Technological Education;Brookshire, Robert G., Ph.D.
Advanced Technological Education;Bayoumi, Abdel-Moez E.
Advanced Technological Education;Devereux, Emily
Advanced Technological Education;Dougal, Roger A.
Environmental Convergence Opportunities in Chemical, Bioengineering, Environmental, and Transport Systems;Berge, Nicole D.
Environmental Convergence Opportunities in Chemical, Bioengineering, Environmental, and Transport Systems;Chaudhry, Hanif
Collaborative Research in Computational Neuroscience;Caicedo, Juan
Collaborative Research in Computational Neuroscience;Chao, Yuh J.
Collaborative Research in Computational Neuroscience;Buell, Duncan A.
National Artificial Intelligence (AI) Research Institutes;Boltin, Nicholas D.
National Artificial Intelligence (AI) Research Institutes;Eastman, Caroline M.
National Artificial Intelligence (AI) Research Institutes;Agostinelli, Forest
Computer and Information Science and Engineering (CISE): Core Programs;Eastman, Caroline M.
Computer and Information Science and Engineering (CISE): Core Programs;Buell, Duncan A.
Division of Materials Research: Topical Materials Research Programs: Biomaterials (BMAT), Condensed Matter Physics (CMP), Metals and Metallic Nanostructures (MMN), Polymers (POL) (nsf20589); Ammal, Salai C.
Division of Materials Research: Topical Materials Research Programs: Biomaterials (BMAT), Condensed Matter Physics (CMP), Metals and Metallic Nanostructures (MMN), Polymers (POL) (nsf20589);Bakos, Jason D.
Division of Materials Research: Topical Materials Research Programs: Ceramics (CER), Electronic and Photonic Materials (EPM), Solid State and Materials Chemistry (SSMC);Ahmad, Iftikhar
Division of Materials Research: Topical Materials Research Programs: Ceramics (CER), Electronic and Photonic Materials (EPM), Solid State and Materials Chemistry (SSMC);Besmann, Theodore  M.
Division of Materials Research: Topical Materials Research Programs: Ceramics (CER), Electronic and Photonic Materials (EPM), Solid State and Materials Chemistry (SSMC);Bayoumi, Abdel-Moez E.
Re-entry to Active Research Program;Besmann, Theodore M.
Re-entry to Active Research Program;Chen, Fanglin (Frank)
Research Experiences for Teachers (RET) in Engineering and Computer Science (nsf20584);Downey, Austin
Research Experiences for Teachers (RET) in Engineering and Computer Science (nsf20584);Bayoumi, Abdel-Moez E.
Condensed Matter and Materials Theory (CMMT) (nsf20582);Ali, Mohammod
Condensed Matter and Materials Theory (CMMT) (nsf20582);Bayat, Mahmoud
Division of Chemistry: Disciplinary Research Programs;Ali, Mohammod
Division of Chemistry: Disciplinary Research Programs;Chen, Fanglin (Frank)
Joint DMS/NIGMS Initiative to Support Research at the Interface of the Biological and Mathematical Sciences;Alexeev, Oleg S.
Joint DMS/NIGMS Initiative to Support Research at the Interface of the Biological and Mathematical Sciences;Carver, Wayne
Joint DMS/NIGMS Initiative to Support Research at the Interface of the Biological and Mathematical Sciences;Caicedo, Juan
Centers for Chemical Innovation;Ali, Mohammod
Centers for Chemical Innovation;Cheng, Xu
NSF Program on Fairness in Artificial Intelligence in Collaboration with Amazon;Agostinelli, Forest
NSF Program on Fairness in Artificial Intelligence in Collaboration with Amazon;Boltin, Nicholas D.
Opportunities for Promoting Understanding through Synthesis;Ali, Mohammod
Opportunities for Promoting Understanding through Synthesis;Bayoumi, Abdel-Moez E.
Cyber-Physical Systems;Boltin, Nicholas D.
Cyber-Physical Systems;Dougal, Roger A.
Hydrologic Sciences (HS);Chaudhry, Hanif
Hydrologic Sciences (HS);Chao, Yuh J.
Spectrum Innovation Initiative: National Center for Wireless Spectrum Research;Crichigno, Jorge
Spectrum Innovation Initiative: National Center for Wireless Spectrum Research;Ali, Mohammod
Gen-4 Engineering Research Centers;Bayoumi, Abdel-Moez E.
Gen-4 Engineering Research Centers;Ahmad, Iftikhar
Future Manufacturing;Bayoumi, Abdel-Moez E.
Future Manufacturing;Ahmad, Iftikhar
Expeditions in Computing (Expeditions) (nsf20544);Agostinelli, Forest
Expeditions in Computing (Expeditions) (nsf20544);Buell, Duncan A.
Reproducible Cells and Organoids via Directed-Differentiation Encoding;Carver, Wayne
Reproducible Cells and Organoids via Directed-Differentiation Encoding;Fan, Daping
NSF-Simons Research Collaborations on the Mathematical and Scientific Foundations of Deep Learning;Agostinelli, Forest
NSF-Simons Research Collaborations on the Mathematical and Scientific Foundations of Deep Learning;Boltin, Nicholas D.
Computer Science for All;Eastman, Caroline M.
Computer Science for All;Buell, Duncan A.
National Robotics Initiative 2.0: Ubiquitous Collaborative Robots;Bayoumi, Abdel-Moez E.
National Robotics Initiative 2.0: Ubiquitous Collaborative Robots;Ali, Mohammod
Environmental Convergence Opportunities in Chemical, Bioengineering, Environmental, and Transport Systems;Berge, Nicole D.
Environmental Convergence Opportunities in Chemical, Bioengineering, Environmental, and Transport Systems;Chaudhry, Hanif
Navigating the New Arctic (NNA) (nsf20514);Ali, Mohammod
Navigating the New Arctic (NNA) (nsf20514);Bayoumi, Abdel-Moez E.
Understanding the Rules of Life: Microbiome Theory and Mechanisms (URoL:MTM) (nsf20513);Ali, Mohammod
Understanding the Rules of Life: Microbiome Theory and Mechanisms (URoL:MTM) (nsf20513);Carver, Wayne
Addressing Systems Challenges through Engineering Teams (ASCENT) (nsf20511);Ali, Mohammod
Addressing Systems Challenges through Engineering Teams (ASCENT) (nsf20511);Dougal, Roger A.
NSF Engineering - UKRI Engineering and Physical Sciences Research Council Lead Agency Opportunity;Ali, Mohammod
NSF Engineering - UKRI Engineering and Physical Sciences Research Council Lead Agency Opportunity;Brookshire, Robert G., Ph.D.
Transitions to Excellence in Molecular and Cellular Biosciences Research (Transitions) (nsf20505);Ali, Mohammod
Transitions to Excellence in Molecular and Cellular Biosciences Research (Transitions) (nsf20505);Carver, Wayne
National Artificial Intelligence (AI) Research Institutes;Boltin, Nicholas D.
National Artificial Intelligence (AI) Research Institutes;Eastman, Caroline M.
NSF/CASIS Collaboration on Transport Phenomena Research on the International Space Station (ISS) to Benefit Life on Earth;Chen, Fanglin (Frank)
NSF/CASIS Collaboration on Transport Phenomena Research on the International Space Station (ISS) to Benefit Life on Earth;Fan, Daping
NSF/CASIS Collaboration on Tissue Engineering and Mechanobiology on the International Space Station (ISS) to Benefit Life on Earth (nsf20500);Bischoff, Jeff
NSF/CASIS Collaboration on Tissue Engineering and Mechanobiology on the International Space Station (ISS) to Benefit Life on Earth (nsf20500);Fan, Daping
IUSE / Professional Formation of Engineers;Bayoumi, Abdel-Moez E.
IUSE / Professional Formation of Engineers;Ali, Mohammod
Formal Methods in the Field;Buell, Duncan A.
Formal Methods in the Field;Dougal, Roger A.
EMERGING FRONTIERS IN RESEARCH AND INNOVATION;Bayoumi, Abdel-Moez E.
EMERGING FRONTIERS IN RESEARCH AND INNOVATION;Ali, Mohammod
NSF/DOE Partnership in Basic Plasma Science and Engineering;Ahmad, Iftikhar
NSF/DOE Partnership in Basic Plasma Science and Engineering;Ali, Mohammod
Computer and Information Science and Engineering (CISE): Core Programs;Eastman, Caroline M.
Computer and Information Science and Engineering (CISE): Core Programs;Buell, Duncan A.
Division of;Ali, Mohammod
Division of;Chen, Fanglin (Frank)
Methodology, Measurement, and Statistics;Devereux, Emily
Methodology, Measurement, and Statistics;Boltin, Nicholas D.
Secure and Trustworthy Cyberspace Frontiers;Eastman, Caroline M.
Secure and Trustworthy Cyberspace Frontiers;Buell, Duncan A.
NSF Program on Fairness in Artificial Intelligence in Collaboration with Amazon (FAI) (nsf19571);Agostinelli, Forest
NSF Program on Fairness in Artificial Intelligence in Collaboration with Amazon (FAI) (nsf19571);Boltin, Nicholas D.
Cyber-Physical Systems;Boltin, Nicholas D.
Cyber-Physical Systems;Dougal, Roger A.
National Robotics Initiative 2.0: Ubiquitous Collaborative Robots (NRI-2.0) (nsf19536);Bayoumi, Abdel-Moez E.
National Robotics Initiative 2.0: Ubiquitous Collaborative Robots (NRI-2.0) (nsf19536);Ali, Mohammod
Enabling Quantum Leap: Quantum Idea Incubator for Transformational Advances in Quantum Systems (QII - TAQS) (nsf19532);Ali, Mohammod
Enabling Quantum Leap: Quantum Idea Incubator for Transformational Advances in Quantum Systems (QII - TAQS) (nsf19532);Ahmad, Iftikhar
Spectrum Efficiency, Energy Efficiency, and Security (SpecEES);Ali, Mohammod
Spectrum Efficiency, Energy Efficiency, and Security (SpecEES);Crichigno, Jorge
CNH2: Dynamics of Integrated Socio-Environmental Systems;Ahmad, Iftikhar
CNH2: Dynamics of Integrated Socio-Environmental Systems;Ali, Mohammod
NSF/CASIS Collaboration on Transport Phenomena Research on the International Space Station (ISS) to Benefit Life on Earth;Chen, Fanglin (Frank)
NSF/CASIS Collaboration on Transport Phenomena Research on the International Space Station (ISS) to Benefit Life on Earth;Fan, Daping
Methodology, Measurement, and Statistics (MMS) (nsf19520);Devereux, Emily
Methodology, Measurement, and Statistics (MMS) (nsf19520);Boltin, Nicholas D.
Designing Materials to Revolutionize and Engineer our Future (DMREF) (nsf19516);Ahmad, Iftikhar
Designing Materials to Revolutionize and Engineer our Future (DMREF) (nsf19516);Ali, Mohammod
Navigating the New Arctic;Ali, Mohammod
Navigating the New Arctic;Bayoumi, Abdel-Moez E.
NSF/CASIS Collaboration on Tissue Engineering and Mechanobiology on the International Space Station (ISS) to Benefit Life on Earth (nsf19509);Bischoff, Jeff
NSF/CASIS Collaboration on Tissue Engineering and Mechanobiology on the International Space Station (ISS) to Benefit Life on Earth (nsf19509);Fan, Daping
Scalable Parallelism in the Extreme (SPX) (nsf19505);Agostinelli, Forest
Scalable Parallelism in the Extreme (SPX) (nsf19505);Buell, Duncan A.
Big Data Regional Innovation Hubs;Bayoumi, Abdel-Moez E.
Big Data Regional Innovation Hubs;Boltin, Nicholas D.
Formal Methods in the Field;Buell, Duncan A.
Formal Methods in the Field;Dougal, Roger A.
Infrastructure Innovation for Biological Research;Bayoumi, Abdel-Moez E.
Infrastructure Innovation for Biological Research;Caicedo, Juan
Collaborative Research in Computational Neuroscience;Caicedo, Juan
Collaborative Research in Computational Neuroscience;Chao, Yuh J.
Division of Molecular and Cellular Biosciences: Investigator-initiated research projects (MCB) (nsf18585);Ali, Mohammod
Division of Molecular and Cellular Biosciences: Investigator-initiated research projects (MCB) (nsf18585);Carver, Wayne
Secure and Trustworthy Cyberspace;Eastman, Caroline M.
Secure and Trustworthy Cyberspace;Buell, Duncan A.
Advanced Technological Education;Brookshire, Robert G., Ph.D.
Advanced Technological Education;Bayoumi, Abdel-Moez E.
Information and Intelligent Systems (IIS): Core Programs;Agostinelli, Forest
Information and Intelligent Systems (IIS): Core Programs;Boltin, Nicholas D.
Computer and Network Systems (CNS): Core Programs;Agostinelli, Forest
Computer and Network Systems (CNS): Core Programs;Boltin, Nicholas D.
Office of Advanced Cyberinfrastructure (OAC): Research Core Program;Agostinelli, Forest
Office of Advanced Cyberinfrastructure (OAC): Research Core Program;Buell, Duncan A.
Joint DMS/NIGMS Initiative to Support Research at the Interface of the Biological and Mathematical Sciences;Alexeev, Oleg S.
Joint DMS/NIGMS Initiative to Support Research at the Interface of the Biological and Mathematical Sciences;Carver, Wayne
Advanced Technological Education (ATE) (nsf17568);Brookshire, Robert G., Ph.D.
Advanced Technological Education (ATE) (nsf17568);Bayoumi, Abdel-Moez E.
Geophysics;Banerjee, Sourav
Geophysics;Ahmad, Iftikhar
Mathematical Sciences Research Institutes (nsf17553);Buell, Duncan A.
Mathematical Sciences Research Institutes (nsf17553);Eastman, Caroline M.
Ideas Lab: Practical Fully-Connected Quantum Computer Challenge (PFCQC);Agostinelli, Forest
Ideas Lab: Practical Fully-Connected Quantum Computer Challenge (PFCQC);Boltin, Nicholas D.
Robert Noyce Teacher Scholarship Program (nsf17541);Brookshire, Robert G., Ph.D.
Robert Noyce Teacher Scholarship Program (nsf17541);Devereux, Emily
Building Community and Capacity in Data Intensive Research in Education;Bayoumi, Abdel-Moez E.
Building Community and Capacity in Data Intensive Research in Education;Dougal, Roger A.
Innovations at the Nexus of Food, Energy and Water Systems;Bayoumi, Abdel-Moez E.
Innovations at the Nexus of Food, Energy and Water Systems;Chaudhry, Hanif
Computer Science for All;Eastman, Caroline M.
Computer Science for All;Buell, Duncan A.
Integrative Strategies for Understanding Neural and Cognitive Systems;Caicedo, Juan
Integrative Strategies for Understanding Neural and Cognitive Systems;Chao, Yuh J.
'''

In [ ]:
df = pd.DataFrame(columns=['Proposal Name', 'Professor Recommended'])
# Create a list to store the data
new_rows = []

for line in content.split('\n')[2:]:
  if line.strip():
    proposal, professor = line.split(';')
    # Append the data to the list
    new_rows.append({'Proposal Name': proposal, 'Professor Recommended': professor})

df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

In [ ]:
df.head()

,Proposal Name,Professor Recommended
0,Advanced Technological Education,"Brookshire, Robert G., Ph.D."
1,Advanced Technological Education,"Bayoumi, Abdel-Moez E."
2,Advanced Technological Education,"Devereux, Emily"
3,Advanced Technological Education,"Dougal, Roger A."
4,Environmental Convergence Opportunities in Che...,"Berge, Nicole D."


In [ ]:
df.to_csv('chatgpt_proposals_professor_recommendations_2.csv', index=False)